<a href="https://colab.research.google.com/github/EmanueleDeCandia/Logistica-Vending-Machine/blob/main/Scenari_OCS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Scenari OCS**

Metodo multi dimensionale con variabili discrete

In [ ]:
from pathlib import Path

script = r'''"""
Simulatore scenari logistici OCS
================================

Genera:
1. Tutte le combinazioni tra:
   - km annui per addetto;
   - clienti per addetto;
   - frequenza media mensile dei rifornimenti.
2. Matrici a doppia entrata:
   - km annui per cliente;
   - km mensili per cliente;
   - km medi per rifornimento, una matrice per ogni scenario di frequenza;
   - rifornimenti annui per addetto;
   - addetti teorici necessari.
3. Un file Excel con tutti i risultati.

Dipendenze:
    pip install pandas numpy openpyxl matplotlib
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# 1. PARAMETRI MODIFICABILI
# ---------------------------------------------------------------------

CLIENTI_OCS = 8_500
ADDETTI_OCS_ATTUALI = 80

SCENARI_KM_ANNO_ADDETTO = [35_000, 40_000, 45_000, 50_000, 55_000, 60_000]
SCENARI_CLIENTI_ADDETTO = [85, 90, 95, 100, 105, 110, 120, 130, 140, 150]


@dataclass(frozen=True)
class ScenarioFrequenza:
    codice: str
    descrizione: str
    quota_1: float = 0.0
    quota_2: float = 0.0
    quota_3: float = 0.0
    quota_4: float = 0.0

    @property
    def frequenza_media_mese(self) -> float:
        """Numero medio ponderato di rifornimenti per cliente al mese."""
        return (
            self.quota_1 * 1
            + self.quota_2 * 2
            + self.quota_3 * 3
            + self.quota_4 * 4
        )

    def valida(self) -> None:
        totale = self.quota_1 + self.quota_2 + self.quota_3 + self.quota_4
        if not np.isclose(totale, 1.0):
            raise ValueError(
                f"Le quote dello scenario {self.codice} sommano a {totale:.4f}, non a 1."
            )
        if self.frequenza_media_mese <= 0:
            raise ValueError(f"La frequenza dello scenario {self.codice} deve essere positiva.")


SCENARI_FREQUENZA = [
    ScenarioFrequenza(
        "F1",
        "100% clienti con 1 rifornimento/mese",
        quota_1=1.00,
    ),
    ScenarioFrequenza(
        "F2",
        "50% con 1 e 50% con 2 rifornimenti/mese",
        quota_1=0.50,
        quota_2=0.50,
    ),
    ScenarioFrequenza(
        "F3",
        "50% con 1, 25% con 2 e 25% con 3 rifornimenti/mese",
        quota_1=0.50,
        quota_2=0.25,
        quota_3=0.25,
    ),
    ScenarioFrequenza(
        "F4",
        "50% con 1, 20% con 2, 20% con 3 e 10% con 4 rifornimenti/mese",
        quota_1=0.50,
        quota_2=0.20,
        quota_3=0.20,
        quota_4=0.10,
    ),
    ScenarioFrequenza(
        "F5",
        "100% clienti con 2 rifornimenti/mese",
        quota_2=1.00,
    ),
]


# ---------------------------------------------------------------------
# 2. CALCOLO DI UNA SINGOLA COMBINAZIONE
# ---------------------------------------------------------------------

def calcola_scenario(
    km_annui_addetto: float,
    clienti_addetto: float,
    frequenza_media_mese: float,
    clienti_totali: int = CLIENTI_OCS,
    addetti_attuali: int = ADDETTI_OCS_ATTUALI,
) -> dict:
    """
    Calcola gli indicatori di una singola combinazione.

    Nota metodologica:
    'Km per rifornimento' è un indicatore medio allocato:
        km mensili per addetto /
        rifornimenti mensili effettuati dall'addetto.

    Non rappresenta necessariamente la distanza fisica tra due clienti,
    perché un giro può comprendere più clienti.
    """
    if km_annui_addetto <= 0:
        raise ValueError("I km annui per addetto devono essere positivi.")
    if clienti_addetto <= 0:
        raise ValueError("I clienti per addetto devono essere positivi.")
    if frequenza_media_mese <= 0:
        raise ValueError("La frequenza mensile deve essere positiva.")

    km_mese_addetto = km_annui_addetto / 12
    km_anno_cliente = km_annui_addetto / clienti_addetto
    km_mese_cliente = km_anno_cliente / 12

    rifornimenti_mese_addetto = clienti_addetto * frequenza_media_mese
    rifornimenti_anno_addetto = rifornimenti_mese_addetto * 12

    km_per_rifornimento = km_mese_addetto / rifornimenti_mese_addetto

    addetti_teorici = clienti_totali / clienti_addetto
    addetti_necessari_interi = int(np.ceil(addetti_teorici))
    differenza_addetti = addetti_necessari_interi - addetti_attuali

    capacita_clienti_con_addetti_attuali = addetti_attuali * clienti_addetto
    clienti_non_coperti = max(
        0,
        clienti_totali - capacita_clienti_con_addetti_attuali,
    )
    clienti_eccedenti_capacita = max(
        0,
        capacita_clienti_con_addetti_attuali - clienti_totali,
    )

    km_totali_flottа_teorica = km_annui_addetto * addetti_teorici
    km_totali_con_addetti_interi = km_annui_addetto * addetti_necessari_interi

    return {
        "km_annui_addetto": km_annui_addetto,
        "km_mese_addetto": km_mese_addetto,
        "clienti_addetto": clienti_addetto,
        "frequenza_media_mese": frequenza_media_mese,
        "km_anno_cliente": km_anno_cliente,
        "km_mese_cliente": km_mese_cliente,
        "rifornimenti_mese_addetto": rifornimenti_mese_addetto,
        "rifornimenti_anno_addetto": rifornimenti_anno_addetto,
        "km_per_rifornimento": km_per_rifornimento,
        "addetti_teorici": addetti_teorici,
        "addetti_necessari_interi": addetti_necessari_interi,
        "differenza_vs_addetti_attuali": differenza_addetti,
        "capacita_clienti_con_addetti_attuali": capacita_clienti_con_addetti_attuali,
        "clienti_non_coperti": clienti_non_coperti,
        "clienti_eccedenti_capacita": clienti_eccedenti_capacita,
        "km_totali_flotta_teorica": km_totali_flottа_teorica,
        "km_totali_con_addetti_interi": km_totali_con_addetti_interi,
    }


# ---------------------------------------------------------------------
# 3. GENERAZIONE DI TUTTI GLI SCENARI
# ---------------------------------------------------------------------

def genera_tutti_scenari() -> pd.DataFrame:
    righe = []

    for frequenza in SCENARI_FREQUENZA:
        frequenza.valida()

        for km_annui in SCENARI_KM_ANNO_ADDETTO:
            for clienti_addetto in SCENARI_CLIENTI_ADDETTO:
                risultato = calcola_scenario(
                    km_annui_addetto=km_annui,
                    clienti_addetto=clienti_addetto,
                    frequenza_media_mese=frequenza.frequenza_media_mese,
                )

                risultato.update(
                    {
                        "scenario_frequenza": frequenza.codice,
                        "descrizione_frequenza": frequenza.descrizione,
                    }
                )
                righe.append(risultato)

    colonne_iniziali = [
        "scenario_frequenza",
        "descrizione_frequenza",
        "frequenza_media_mese",
        "km_annui_addetto",
        "clienti_addetto",
    ]

    df = pd.DataFrame(righe)
    altre_colonne = [c for c in df.columns if c not in colonne_iniziali]
    return df[colonne_iniziali + altre_colonne].sort_values(
        ["scenario_frequenza", "km_annui_addetto", "clienti_addetto"]
    )


# ---------------------------------------------------------------------
# 4. MATRICI A DOPPIA ENTRATA
# ---------------------------------------------------------------------

def crea_matrice(
    df: pd.DataFrame,
    variabile_risultato: str,
    scenario_frequenza: str | None = None,
) -> pd.DataFrame:
    """
    Righe: km annui per addetto
    Colonne: clienti per addetto
    Celle: variabile di risultato scelta
    """
    dati = df.copy()

    if scenario_frequenza is not None:
        dati = dati[dati["scenario_frequenza"] == scenario_frequenza]

    matrice = dati.pivot_table(
        index="km_annui_addetto",
        columns="clienti_addetto",
        values=variabile_risultato,
        aggfunc="first",
    )

    matrice.index.name = "Km annui per addetto"
    matrice.columns.name = "Clienti per addetto"
    return matrice


def crea_matrici(df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    matrici: Dict[str, pd.DataFrame] = {}

    # Queste matrici non dipendono dalla frequenza.
    df_base = df[df["scenario_frequenza"] == "F1"]

    matrici["km_annui_per_cliente"] = crea_matrice(
        df_base,
        "km_anno_cliente",
    )
    matrici["km_mensili_per_cliente"] = crea_matrice(
        df_base,
        "km_mese_cliente",
    )
    matrici["addetti_teorici"] = crea_matrice(
        df_base,
        "addetti_teorici",
    )
    matrici["addetti_necessari_interi"] = crea_matrice(
        df_base,
        "addetti_necessari_interi",
    )

    # Queste matrici cambiano per scenario di frequenza.
    for frequenza in SCENARI_FREQUENZA:
        codice = frequenza.codice

        matrici[f"km_per_rifornimento_{codice}"] = crea_matrice(
            df,
            "km_per_rifornimento",
            scenario_frequenza=codice,
        )
        matrici[f"rifornimenti_annui_addetto_{codice}"] = crea_matrice(
            df,
            "rifornimenti_anno_addetto",
            scenario_frequenza=codice,
        )
        matrici[f"km_totali_flotta_{codice}"] = crea_matrice(
            df,
            "km_totali_con_addetti_interi",
            scenario_frequenza=codice,
        )

    return matrici


# ---------------------------------------------------------------------
# 5. ANALISI DI UNO SCENARIO SPECIFICO
# ---------------------------------------------------------------------

def seleziona_scenario(
    df: pd.DataFrame,
    km_annui_addetto: int,
    clienti_addetto: int,
    scenario_frequenza: str,
) -> pd.Series:
    filtro = (
        (df["km_annui_addetto"] == km_annui_addetto)
        & (df["clienti_addetto"] == clienti_addetto)
        & (df["scenario_frequenza"] == scenario_frequenza)
    )

    risultato = df.loc[filtro]

    if risultato.empty:
        raise KeyError("La combinazione richiesta non esiste.")
    if len(risultato) > 1:
        raise RuntimeError("La combinazione richiesta non è univoca.")

    return risultato.iloc[0]


# ---------------------------------------------------------------------
# 6. ESPORTAZIONE EXCEL
# ---------------------------------------------------------------------

def esporta_excel(
    df: pd.DataFrame,
    matrici: Dict[str, pd.DataFrame],
    percorso: str | Path = "risultati_scenari_ocs.xlsx",
) -> Path:
    percorso = Path(percorso)

    frequenze_df = pd.DataFrame(
        [
            {
                "codice": f.codice,
                "descrizione": f.descrizione,
                "quota_1": f.quota_1,
                "quota_2": f.quota_2,
                "quota_3": f.quota_3,
                "quota_4": f.quota_4,
                "frequenza_media_mese": f.frequenza_media_mese,
            }
            for f in SCENARI_FREQUENZA
        ]
    )

    with pd.ExcelWriter(percorso, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="Tutti_scenari", index=False)
        frequenze_df.to_excel(writer, sheet_name="Frequenze", index=False)

        for nome, matrice in matrici.items():
            # Excel limita il nome del foglio a 31 caratteri.
            nome_foglio = nome[:31]
            matrice.to_excel(writer, sheet_name=nome_foglio)

    return percorso.resolve()


# ---------------------------------------------------------------------
# 7. HEATMAP OPZIONALE
# ---------------------------------------------------------------------

def salva_heatmap(
    matrice: pd.DataFrame,
    titolo: str,
    percorso: str | Path,
    decimali: int = 1,
) -> Path:
    """
    Salva una heatmap con matplotlib.
    Non è necessaria per il calcolo; serve solo per la visualizzazione.
    """
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 6))
    immagine = ax.imshow(matrice.values, aspect="auto")

    ax.set_xticks(range(len(matrice.columns)))
    ax.set_xticklabels(matrice.columns)
    ax.set_yticks(range(len(matrice.index)))
    ax.set_yticklabels(matrice.index)

    ax.set_xlabel("Clienti per addetto")
    ax.set_ylabel("Km annui per addetto")
    ax.set_title(titolo)

    for riga in range(matrice.shape[0]):
        for colonna in range(matrice.shape[1]):
            valore = matrice.iloc[riga, colonna]
            ax.text(
                colonna,
                riga,
                f"{valore:.{decimali}f}",
                ha="center",
                va="center",
            )

    fig.colorbar(immagine, ax=ax)
    fig.tight_layout()

    percorso = Path(percorso)
    fig.savefig(percorso, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return percorso.resolve()


# ---------------------------------------------------------------------
# 8. ESECUZIONE
# ---------------------------------------------------------------------

def main() -> None:
    df = genera_tutti_scenari()
    matrici = crea_matrici(df)

    print(f"Combinazioni generate: {len(df)}")
    print()

    scenario_a = seleziona_scenario(
        df,
        km_annui_addetto=50_000,
        clienti_addetto=150,
        scenario_frequenza="F1",
    )

    print("SIMULAZIONE A")
    print(scenario_a.to_string())
    print()

    print("MATRICE KM MEDI PER RIFORNIMENTO – F1")
    print(matrici["km_per_rifornimento_F1"].round(2))
    print()

    file_excel = esporta_excel(
        df,
        matrici,
        percorso="risultati_scenari_ocs.xlsx",
    )
    print(f"File Excel creato: {file_excel}")

    # Esempio facoltativo di heatmap.
    salva_heatmap(
        matrici["km_per_rifornimento_F1"],
        titolo="Km medi per rifornimento – scenario F1",
        percorso="heatmap_km_per_rifornimento_F1.png",
        decimali=2,
    )


if __name__ == "__main__":
    main()
'''

# Fix a unicode lookalike typo in variable name before saving.
script = script.replace("km_totali_flottа_teorica", "km_totali_flotta_teorica")

path = Path("/mnt/data/simulatore_scenari_ocs.py")
# Create the parent directory if it doesn't exist
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(script, encoding="utf-8")

print(path)
print(f"Righe: {len(script.splitlines())}")

/mnt/data/simulatore_scenari_ocs.py
Righe: 456


In [ ]:
import subprocess, sys, os, textwrap, json, pathlib, pandas as pd

result = subprocess.run(
    [sys.executable, "/mnt/data/simulatore_scenari_ocs.py"],
    cwd="/mnt/data",
    capture_output=True,
    text=True,
    timeout=120,
)
print(result.stdout)
print(result.stderr)
print("returncode:", result.returncode)


Combinazioni generate: 300

SIMULAZIONE A
scenario_frequenza                                                        F1
descrizione_frequenza                   100% clienti con 1 rifornimento/mese
frequenza_media_mese                                                     1.0
km_annui_addetto                                                       50000
clienti_addetto                                                          150
km_mese_addetto                                                  4166.666667
km_anno_cliente                                                   333.333333
km_mese_cliente                                                    27.777778
rifornimenti_mese_addetto                                              150.0
rifornimenti_anno_addetto                                             1800.0
km_per_rifornimento                                                27.777778
addetti_teorici                                                    56.666667
addetti_necessari_interi          

In [ ]:
import sys
import pandas as pd # Ensure pandas is available

# Add the directory containing the script to the Python path
if '/mnt/data' not in sys.path:
    sys.path.insert(0, '/mnt/data')

# Import the script as a module
import simulatore_scenari_ocs as scs

# Generate the DataFrame (df) using the function from the imported module
df = scs.genera_tutti_scenari()

# Now, seleziona_scenario is accessible through the scs module
scenario_a = scs.seleziona_scenario(
    df,
    km_annui_addetto=50_000,
    clienti_addetto=130,
    scenario_frequenza="F1",
)

print(scenario_a)

scenario_frequenza                                                        F1
descrizione_frequenza                   100% clienti con 1 rifornimento/mese
frequenza_media_mese                                                     1.0
km_annui_addetto                                                       50000
clienti_addetto                                                          130
km_mese_addetto                                                  4166.666667
km_anno_cliente                                                   384.615385
km_mese_cliente                                                    32.051282
rifornimenti_mese_addetto                                              130.0
rifornimenti_anno_addetto                                             1560.0
km_per_rifornimento                                                32.051282
addetti_teorici                                                    65.384615
addetti_necessari_interi                                                  66

In [ ]:
matrice = scs.crea_matrice(
    df,
    variabile_risultato="km_per_rifornimento",
    scenario_frequenza="F1",
)

print(matrice.round(2))

Clienti per addetto     85     90     95     100    105    110    120    130  \
Km annui per addetto                                                           
35000                 34.31  32.41  30.70  29.17  27.78  26.52  24.31  22.44   
40000                 39.22  37.04  35.09  33.33  31.75  30.30  27.78  25.64   
45000                 44.12  41.67  39.47  37.50  35.71  34.09  31.25  28.85   
50000                 49.02  46.30  43.86  41.67  39.68  37.88  34.72  32.05   
55000                 53.92  50.93  48.25  45.83  43.65  41.67  38.19  35.26   
60000                 58.82  55.56  52.63  50.00  47.62  45.45  41.67  38.46   

Clienti per addetto     140    150  
Km annui per addetto                
35000                 20.83  19.44  
40000                 23.81  22.22  
45000                 26.79  25.00  
50000                 29.76  27.78  
55000                 32.74  30.56  
60000                 35.71  33.33  


In [ ]:
# SCENARI_KM_ANNO_ADDETTO = [35_000, 40_000, 45_000, 50_000, 55_000, 60_000]
# SCENARI_CLIENTI_ADDETTO = [85, 90, 95, 100, 105, 110, 120, 130, 140, 150]
# SCENARI_FREQUENZA = [
#     # F1: 1.0 rifornimento/mese
#     # F2: 1.5 rifornimenti/mese
#     # F3: 1.75 rifornimenti/mese
#     # F4: 1.8 rifornimenti/mese
#     # F5: 2.0 rifornimenti/mese
# ]

matrice = scs.crea_matrice(
    df,
    variabile_risultato="km_per_rifornimento",
    scenario_frequenza="F1",
)

print(matrice.round(2))

Clienti per addetto     85     90     95     100    105    110    120    130  \
Km annui per addetto                                                           
35000                 34.31  32.41  30.70  29.17  27.78  26.52  24.31  22.44   
40000                 39.22  37.04  35.09  33.33  31.75  30.30  27.78  25.64   
45000                 44.12  41.67  39.47  37.50  35.71  34.09  31.25  28.85   
50000                 49.02  46.30  43.86  41.67  39.68  37.88  34.72  32.05   
55000                 53.92  50.93  48.25  45.83  43.65  41.67  38.19  35.26   
60000                 58.82  55.56  52.63  50.00  47.62  45.45  41.67  38.46   

Clienti per addetto     140    150  
Km annui per addetto                
35000                 20.83  19.44  
40000                 23.81  22.22  
45000                 26.79  25.00  
50000                 29.76  27.78  
55000                 32.74  30.56  
60000                 35.71  33.33  


Ora eseguiamo la funzione `main()` dello script `simulatore_scenari_ocs.py`. Questa funzione gestirà l'intero processo:
1.  Genererà tutte le combinazioni di scenari possibili.
2.  Calcolerà tutte le metriche e creerà le matrici a doppia entrata.
3.  Esportà tutti i risultati in un singolo file Excel chiamato `risultati_scenari_ocs.xlsx` nella directory `/mnt/data/`.
4.  Genererà anche una heatmap di esempio per i 'Km medi per rifornimento – scenario F1'.

In [ ]:
# Esegui la funzione main per generare tutti gli scenari, le matrici e il file Excel.
scs.main()

Combinazioni generate: 300

SIMULAZIONE A
scenario_frequenza                                                        F1
descrizione_frequenza                   100% clienti con 1 rifornimento/mese
frequenza_media_mese                                                     1.0
km_annui_addetto                                                       50000
clienti_addetto                                                          150
km_mese_addetto                                                  4166.666667
km_anno_cliente                                                   333.333333
km_mese_cliente                                                    27.777778
rifornimenti_mese_addetto                                              150.0
rifornimenti_anno_addetto                                             1800.0
km_per_rifornimento                                                27.777778
addetti_teorici                                                    56.666667
addetti_necessari_interi          

### Seleziona e Visualizza Scenari Interattivamente

Modifica i valori qui sotto per esplorare diversi scenari e le relative matrici.

In [ ]:
# Definisci le variabili per lo scenario che vuoi esplorare
km_annui_scelto = 50_000  # Prova con 35_000, 40_000, 45_000, 50_000, 55_000, 60_000
clienti_scelto = 95       # Prova con 85, 90, 95, 100, 105, 110, 120, 130, 140, 150
frequenza_scelta = "F3"   # Prova con "F1", "F2", "F3", "F4", "F5"

print(f"--- Dettagli per lo scenario selezionato ({km_annui_scelto} km/anno, {clienti_scelto} clienti, Frequenza {frequenza_scelta}) ---")
scenario_selezionato = scs.seleziona_scenario(
    df,
    km_annui_addetto=km_annui_scelto,
    clienti_addetto=clienti_scelto,
    scenario_frequenza=frequenza_scelta,
)
print(scenario_selezionato.to_string())

print(f"\n--- Matrice 'Km per rifornimento' per la frequenza {frequenza_scelta} ---")
matrice_selezionata = scs.crea_matrice(
    df,
    variabile_risultato="km_per_rifornimento",
    scenario_frequenza=frequenza_scelta,
)
print(matrice_selezionata.round(2))

--- Dettagli per lo scenario selezionato (50000 km/anno, 95 clienti, Frequenza F3) ---
scenario_frequenza                                                                     F3
descrizione_frequenza                   50% con 1, 25% con 2 e 25% con 3 rifornimenti/...
frequenza_media_mese                                                                 1.75
km_annui_addetto                                                                    50000
clienti_addetto                                                                        95
km_mese_addetto                                                               4166.666667
km_anno_cliente                                                                526.315789
km_mese_cliente                                                                 43.859649
rifornimenti_mese_addetto                                                          166.25
rifornimenti_anno_addetto                                                          1995.0
km_per_riforn

# Guida all’uso del simulatore logistico ed economico OCS

## 1. Obiettivo del modello

Questo modello serve a simulare diversi scenari di organizzazione della logistica OCS, combinando variabili operative ed economiche.

Le principali variabili considerate sono:

* chilometri annui percorsi da ciascun addetto;
* numero di clienti serviti da ciascun addetto;
* frequenza mensile dei rifornimenti;
* valore economico annuo medio di ciascun cliente;
* costo totale per chilometro;
* numero complessivo di clienti OCS;
* numero attuale di addetti OCS;
* giorni lavorativi medi mensili.

Il modello genera tutte le possibili combinazioni tra questi valori e calcola, per ciascuna combinazione:

* produttività operativa;
* fabbisogno di personale;
* chilometri totali;
* costi logistici;
* margini economici;
* incidenza percentuale della logistica sul valore del cliente e sul portafoglio complessivo.

---

# 2. Distinzione fondamentale tra simulazione e matrici standard

Nel codice sono presenti due livelli distinti di analisi.

## 2.1 Funzione `calcola_scenario`

La funzione:

```python
calcola_scenario(...)
```

calcola tutti gli indicatori relativi a una singola combinazione di parametri.

Ogni riga della tabella finale rappresenta quindi uno scenario completo, definito da:

```text
Km annui per addetto
× Clienti per addetto
× Scenario di frequenza
× Valore annuo del cliente
× Costo totale per km
```

Gli indicatori economici di flotta sono calcolati direttamente all’interno di questa funzione e fanno riferimento ai valori presenti nella singola riga.

Non utilizzano automaticamente i valori standard:

```python
40_000 km per addetto
100 clienti per addetto
scenario F1
```

Questi valori sono solo una delle possibili combinazioni presenti nella tabella.

## 2.2 Funzione `crea_matrici_standard`

La funzione:

```python
def crea_matrici_standard(
    df: pd.DataFrame,
    km_scenario_economico: float = 40_000,
    clienti_scenario_economico: float = 100,
    frequenza_scenario_economico: str = "F1",
)
```

non determina i risultati di tutte le simulazioni.

Serve invece a selezionare uno specifico scenario operativo per costruire alcune matrici economiche a doppia entrata.

I valori predefiniti:

```python
km_scenario_economico = 40_000
clienti_scenario_economico = 100
frequenza_scenario_economico = "F1"
```

significano che, se non vengono specificati valori diversi, le matrici economiche confronteranno:

* i diversi valori economici del cliente;
* i diversi costi per chilometro;

mantenendo fissi:

* 40.000 km annui per addetto;
* 100 clienti per addetto;
* scenario F1, cioè un rifornimento mensile per cliente.

Questi valori sono quindi parametri di selezione delle matrici e non ipotesi applicate automaticamente a tutta la simulazione.

---

# 3. Parametri generali del modello

## 3.1 `CLIENTI_OCS`

```python
CLIENTI_OCS = 8_500
```

Rappresenta il numero complessivo di clienti OCS che l’impresa deve servire.

Questo valore è utilizzato per calcolare:

* il numero teorico di addetti necessari;
* il numero intero di addetti necessari;
* il valore economico complessivo del portafoglio;
* i chilometri complessivi della flotta;
* il costo logistico complessivo;
* il margine logistico complessivo.

---

## 3.2 `ADDETTI_OCS_ATTUALI`

```python
ADDETTI_OCS_ATTUALI = 80
```

Rappresenta il numero attuale di addetti dedicati al rifornimento OCS.

È utilizzato per confrontare il fabbisogno teorico di personale con la dotazione attuale.

Permette di calcolare:

* addetti mancanti;
* eventuali addetti eccedenti;
* capacità complessiva di copertura;
* clienti eventualmente non coperti.

---

## 3.3 `GIORNI_LAVORATIVI_MESE`

```python
GIORNI_LAVORATIVI_MESE = 22
```

Rappresenta il numero medio di giorni lavorativi mensili utilizzato per trasformare i valori mensili in valori giornalieri.

È impiegato soprattutto per calcolare:

```python
rifornimenti_giorno_addetto
```

e:

```python
km_giorno_addetto
```

---

# 4. Variabili operative di scenario

## 4.1 `km_annui_addetto`

Rappresenta i chilometri percorsi mediamente in un anno da ciascun addetto.

Gli scenari previsti sono, per esempio:

```python
SCENARI_KM_ANNO_ADDETTO = [
    35_000,
    40_000,
    45_000,
    50_000,
    55_000,
    60_000,
]
```

Questa variabile influenza direttamente:

* km mensili per addetto;
* km giornalieri per addetto;
* km annui per cliente;
* km mensili per cliente;
* km medi per rifornimento;
* costo logistico per addetto;
* costo logistico per cliente;
* costo logistico della flotta.

---

## 4.2 `clienti_addetto`

Rappresenta il numero medio di clienti assegnati a ogni addetto.

Gli scenari previsti sono:

```python
SCENARI_CLIENTI_ADDETTO = [
    85,
    90,
    95,
    100,
    105,
    110,
    120,
    130,
    140,
    150,
]
```

Questa variabile influenza:

* numero di addetti necessari;
* rifornimenti mensili per addetto;
* rifornimenti giornalieri per addetto;
* km annui allocati per cliente;
* km medi per rifornimento;
* valore economico gestito da ciascun addetto;
* costo logistico unitario per cliente.

A parità di chilometri annui, un maggiore numero di clienti per addetto riduce i chilometri allocati mediamente a ciascun cliente.

---

## 4.3 `frequenza_media_mese`

Rappresenta il numero medio di rifornimenti effettuati ogni mese per ciascun cliente.

Non è sempre un numero intero, perché può derivare dalla media ponderata tra clienti con frequenze diverse.

Esempio:

```text
50% dei clienti: 1 rifornimento al mese
50% dei clienti: 2 rifornimenti al mese
```

La frequenza media mensile è:

```text
0,50 × 1 + 0,50 × 2 = 1,50
```

Gli scenari standard sono:

| Scenario | Composizione                               | Frequenza media |
| -------- | ------------------------------------------ | --------------: |
| F1       | 100% dei clienti con 1 rifornimento        |            1,00 |
| F2       | 50% con 1 e 50% con 2                      |            1,50 |
| F3       | 50% con 1, 25% con 2, 25% con 3            |            1,75 |
| F4       | 50% con 1, 20% con 2, 20% con 3, 10% con 4 |            1,90 |
| F5       | 100% dei clienti con 2 rifornimenti        |            2,00 |

La frequenza influenza principalmente:

* rifornimenti mensili per addetto;
* rifornimenti giornalieri per addetto;
* rifornimenti annui per addetto;
* km medi allocati a ciascun rifornimento;
* valore medio attribuito a ciascun rifornimento;
* margine logistico per rifornimento.

---

# 5. Variabili economiche

## 5.1 `valore_annuo_cliente`

Rappresenta il valore economico annuo medio di un cliente.

Nel modello può essere interpretato come:

* ricavo annuo medio per cliente;
* fatturato annuo medio per cliente;
* valore annuo del contratto;
* valore economico annuo servito dalla logistica.

Gli scenari previsti possono essere:

```python
SCENARI_VALORE_ANNUO_CLIENTE = [
    100,
    200,
    500,
    1_000,
    2_000,
    5_000,
]
```

È importante sottolineare che il valore è considerato annuale.

Quindi:

```text
Valore cliente = 1.000 €
```

significa:

```text
1.000 € per cliente all’anno
```

Questa variabile è utilizzata per calcolare:

* valore annuo gestito da ciascun addetto;
* valore annuo complessivo del portafoglio;
* margine logistico per cliente;
* margine logistico per addetto;
* margine logistico del portafoglio;
* incidenza percentuale del costo logistico;
* valore medio attribuito a ciascun rifornimento.

---

## 5.2 `costo_km_totale`

Rappresenta il costo complessivo associato a ogni chilometro percorso.

Gli scenari previsti sono:

```python
SCENARI_COSTO_KM_TOTALE = [
    0.30,
    0.40,
    0.50,
    0.60,
    1.00,
    1.50,
    2.00,
]
```

Il costo può comprendere, in funzione del perimetro scelto:

* costo del personale durante il viaggio;
* carburante o energia;
* manutenzione ordinaria;
* pneumatici;
* assicurazione;
* ammortamento o leasing del veicolo;
* costi amministrativi della flotta;
* costi logistici indiretti;
* tempi improduttivi;
* gestione e coordinamento operativo.

Il modello considera questo valore come costo totale per chilometro.

Pertanto, se alcuni costi del personale sono già conteggiati separatamente, bisogna evitare di includerli nuovamente nel costo chilometrico.

---

# 6. Indicatori operativi

## 6.1 `km_mese_addetto`

Formula:

```python
km_mese_addetto = km_annui_addetto / 12
```

Indica quanti chilometri percorre mediamente un addetto in un mese.

Esempio:

```text
40.000 km annui ÷ 12 = 3.333,33 km mensili
```

---

## 6.2 `km_giorno_addetto`

Formula:

```python
km_giorno_addetto = km_mese_addetto / giorni_lavorativi_mese
```

Indica i chilometri medi giornalieri per addetto.

Esempio:

```text
3.333,33 km mensili ÷ 22 giorni = 151,52 km al giorno
```

---

## 6.3 `km_anno_cliente`

Formula:

```python
km_anno_cliente = km_annui_addetto / clienti_addetto
```

Rappresenta la quota media di chilometri annui attribuita a ciascun cliente.

Esempio:

```text
40.000 km annui ÷ 100 clienti = 400 km per cliente all’anno
```

È un indicatore medio di allocazione, non necessariamente la distanza stradale effettiva del cliente.

---

## 6.4 `km_mese_cliente`

Formula:

```python
km_mese_cliente = km_anno_cliente / 12
```

Rappresenta la quota media mensile di chilometri attribuita a ciascun cliente.

Esempio:

```text
400 km annui ÷ 12 = 33,33 km mensili per cliente
```

---

## 6.5 `rifornimenti_mese_addetto`

Formula:

```python
rifornimenti_mese_addetto = clienti_addetto * frequenza_media_mese
```

Rappresenta il numero medio di rifornimenti che un addetto deve effettuare in un mese.

Esempio con 100 clienti e frequenza F1:

```text
100 clienti × 1 rifornimento = 100 rifornimenti al mese
```

Esempio con 100 clienti e frequenza F2:

```text
100 clienti × 1,5 rifornimenti medi = 150 rifornimenti al mese
```

---

## 6.6 `rifornimenti_giorno_addetto`

Formula:

```python
rifornimenti_giorno_addetto = (
    rifornimenti_mese_addetto / giorni_lavorativi_mese
)
```

Rappresenta il numero medio di rifornimenti che ciascun addetto deve effettuare ogni giorno lavorativo.

Esempio:

```text
100 rifornimenti mensili ÷ 22 giorni = 4,55 rifornimenti al giorno
```

Questo indicatore è particolarmente utile per verificare la compatibilità tra:

* numero di rifornimenti;
* durata media di ogni intervento;
* tempi di percorrenza;
* ore lavorative disponibili.

---

## 6.7 `rifornimenti_anno_addetto`

Formula:

```python
rifornimenti_anno_addetto = rifornimenti_mese_addetto * 12
```

Indica il numero medio annuo di rifornimenti effettuati da ciascun addetto.

---

## 6.8 `km_per_rifornimento`

Formula:

```python
km_per_rifornimento = (
    km_mese_addetto / rifornimenti_mese_addetto
)
```

Rappresenta la quota media di chilometri attribuita a ciascun rifornimento.

Esempio:

```text
3.333,33 km mensili ÷ 100 rifornimenti = 33,33 km per rifornimento
```

Questo valore non rappresenta necessariamente la distanza fisica tra la sede e un singolo cliente.

Un giro logistico può infatti comprendere più clienti.

L’indicatore deve essere interpretato come:

```text
chilometri complessivi medi allocati a ogni operazione di rifornimento
```

---

# 7. Indicatori relativi agli addetti

## 7.1 `addetti_teorici`

Formula:

```python
addetti_teorici = clienti_totali / clienti_addetto
```

Indica il numero matematico di addetti necessario per servire tutti i clienti.

Esempio:

```text
8.500 clienti ÷ 100 clienti per addetto = 85 addetti
```

Il risultato può essere decimale.

Esempio:

```text
8.500 ÷ 110 = 77,27 addetti
```

Il valore teorico è utile per le analisi economiche continue, ma non corrisponde necessariamente a un organico concretamente utilizzabile.

---

## 7.2 `addetti_necessari_interi`

Formula:

```python
addetti_necessari_interi = ceil(addetti_teorici)
```

Arrotonda il fabbisogno teorico all’intero superiore.

Esempio:

```text
77,27 addetti teorici → 78 addetti necessari
```

Questa è la misura più prudente per una simulazione operativa, perché non è possibile impiegare una frazione di addetto senza introdurre ipotesi su part-time, condivisione di risorse o attività miste.

---

## 7.3 `differenza_vs_addetti_attuali`

Formula:

```python
differenza_vs_addetti_attuali = (
    addetti_necessari_interi - addetti_attuali
)
```

Confronta il fabbisogno dello scenario con gli addetti attualmente disponibili.

Interpretazione:

* valore positivo: servono addetti aggiuntivi;
* valore pari a zero: l’organico è coerente;
* valore negativo: esiste capacità eccedente.

---

## 7.4 `capacita_clienti_con_addetti_attuali`

Formula:

```python
capacita_clienti_con_addetti_attuali = (
    addetti_attuali * clienti_addetto
)
```

Indica quanti clienti potrebbero essere serviti dagli addetti attuali, assumendo il livello di produttività previsto dallo scenario.

Esempio:

```text
80 addetti × 100 clienti = 8.000 clienti
```

---

## 7.5 `clienti_non_coperti`

Formula:

```python
clienti_non_coperti = max(
    0,
    clienti_totali - capacita_clienti_con_addetti_attuali
)
```

Indica quanti clienti resterebbero non coperti con l’organico attuale.

Esempio:

```text
8.500 clienti totali - 8.000 di capacità = 500 clienti non coperti
```

---

# 8. Indicatori dei chilometri complessivi di flotta

## 8.1 `km_totali_flotta_teorica`

Formula:

```python
km_totali_flotta_teorica = (
    km_annui_addetto * addetti_teorici
)
```

Rappresenta i chilometri annui complessivi della flotta calcolati utilizzando il numero teorico, anche decimale, di addetti.

Esempio:

```text
40.000 km per addetto × 77,27 addetti teorici
= 3.090.909 km
```

Questo indicatore è teorico perché utilizza un numero frazionario di addetti.

È utile per:

* confronti tra scenari;
* analisi economiche continue;
* valutazioni preliminari;
* interpolazioni;
* simulazioni prive del vincolo di indivisibilità del personale.

Non deve essere interpretato automaticamente come il chilometraggio operativo reale.

---

## 8.2 `km_totali_con_addetti_interi`

Formula:

```python
km_totali_con_addetti_interi = (
    km_annui_addetto * addetti_necessari_interi
)
```

Rappresenta i chilometri annui complessivi della flotta assumendo l’impiego del numero intero di addetti necessario.

Esempio:

```text
40.000 km × 78 addetti = 3.120.000 km
```

Questo indicatore è generalmente più vicino alla pianificazione operativa concreta.

Può però sovrastimare leggermente i chilometri se l’ultimo addetto non lavora a piena saturazione.

---

# 9. Indicatori economici unitari

## 9.1 `costo_logistico_annuo_addetto`

Formula:

```python
costo_logistico_annuo_addetto = (
    km_annui_addetto * costo_km_totale
)
```

Rappresenta il costo logistico annuo associato a ciascun addetto.

Esempio:

```text
40.000 km × 0,60 €/km = 24.000 € all’anno
```

---

## 9.2 `costo_logistico_annuo_cliente`

Formula:

```python
costo_logistico_annuo_cliente = (
    km_anno_cliente * costo_km_totale
)
```

Rappresenta il costo logistico medio annuo attribuito a ogni cliente.

Esempio:

```text
400 km per cliente × 0,60 €/km = 240 € all’anno
```

---

## 9.3 `costo_logistico_per_rifornimento`

Formula:

```python
costo_logistico_per_rifornimento = (
    km_per_rifornimento * costo_km_totale
)
```

Rappresenta il costo logistico medio attribuito a ciascun rifornimento.

Esempio:

```text
33,33 km × 0,60 €/km = 20 € per rifornimento
```

---

## 9.4 `valore_annuo_per_addetto`

Formula:

```python
valore_annuo_per_addetto = (
    clienti_addetto * valore_annuo_cliente
)
```

Indica il valore economico annuo del portafoglio gestito da un singolo addetto.

Esempio:

```text
100 clienti × 1.000 € = 100.000 € per addetto
```

---

## 9.5 `margine_logistico_annuo_cliente`

Formula:

```python
margine_logistico_annuo_cliente = (
    valore_annuo_cliente - costo_logistico_annuo_cliente
)
```

Rappresenta il valore residuo del cliente dopo aver sottratto esclusivamente il costo logistico chilometrico.

Esempio:

```text
1.000 € - 240 € = 760 €
```

Non è un margine aziendale completo.

Non considera, salvo che siano già inclusi nel costo chilometrico:

* costo dei prodotti;
* costo delle merci;
* ammortamento delle attrezzature;
* amministrazione;
* attività commerciali;
* assistenza;
* guasti;
* costi generali;
* imposte;
* altri costi del personale.

È quindi più corretto definirlo:

```text
margine dopo il costo logistico considerato dal modello
```

---

## 9.6 `incidenza_costo_logistico_cliente`

Formula:

```python
incidenza_costo_logistico_cliente = (
    costo_logistico_annuo_cliente / valore_annuo_cliente
)
```

Misura la quota del valore annuo del cliente assorbita dal costo logistico.

Esempio:

```text
240 € ÷ 1.000 € = 24%
```

Interpretazione:

* valori bassi: logistica relativamente sostenibile;
* valori elevati: cliente potenzialmente poco efficiente;
* valori superiori al 100%: il solo costo logistico supera il valore annuo del cliente.

---

## 9.7 `margine_logistico_percentuale_cliente`

Formula:

```python
margine_logistico_percentuale_cliente = (
    margine_logistico_annuo_cliente / valore_annuo_cliente
)
```

Indica la percentuale del valore cliente che rimane dopo la sottrazione del costo logistico considerato.

È equivalente a:

```text
1 - incidenza del costo logistico
```

---

## 9.8 `valore_medio_per_rifornimento`

Formula:

```python
valore_medio_per_rifornimento = (
    valore_annuo_cliente
    / (frequenza_media_mese * 12)
)
```

Ripartisce il valore annuo del cliente sul numero medio annuo di rifornimenti.

Esempio con valore cliente pari a 1.000 € e frequenza F1:

```text
1.000 € ÷ 12 rifornimenti = 83,33 € per rifornimento
```

Esempio con frequenza F5:

```text
1.000 € ÷ 24 rifornimenti = 41,67 € per rifornimento
```

Una maggiore frequenza riduce il valore medio associato a ogni visita.

---

## 9.9 `margine_logistico_per_rifornimento`

Formula:

```python
margine_logistico_per_rifornimento = (
    valore_medio_per_rifornimento
    - costo_logistico_per_rifornimento
)
```

Indica il valore medio residuo di ogni rifornimento dopo il costo logistico attribuito.

Anche questo non è un margine economico completo, perché considera esclusivamente il costo logistico incluso nel parametro `costo_km_totale`.

---

# 10. Indicatori economici di flotta

Gli indicatori economici di flotta riportano i risultati dalla singola relazione cliente-addetto all’intero portafoglio OCS.

Per interpretarli correttamente bisogna distinguere due modalità di calcolo:

1. modalità teorica, con addetti anche frazionari;
2. modalità operativa, con addetti arrotondati all’intero superiore.

---

## 10.1 `valore_annuo_portafoglio`

Formula:

```python
valore_annuo_portafoglio = (
    clienti_totali * valore_annuo_cliente
)
```

Rappresenta il valore annuo complessivo di tutti i clienti OCS.

Esempio:

```text
8.500 clienti × 1.000 € = 8.500.000 €
```

Questo indicatore dipende da:

* numero totale dei clienti;
* valore annuo medio per cliente.

Non dipende direttamente:

* dai chilometri per addetto;
* dal numero di clienti per addetto;
* dalla frequenza dei rifornimenti;
* dal costo per km.

---

## 10.2 `costo_logistico_flotta_teorica`

Formula:

```python
costo_logistico_flotta_teorica = (
    km_totali_flotta_teorica * costo_km_totale
)
```

Equivalentemente:

```python
costo_logistico_flotta_teorica = (
    km_annui_addetto
    * addetti_teorici
    * costo_km_totale
)
```

Dipende quindi da:

* clienti totali;
* clienti per addetto;
* km annui per addetto;
* costo totale per km.

Poiché:

```python
addetti_teorici = clienti_totali / clienti_addetto
```

la formula completa è:

```text
Costo flotta teorico =
Km annui per addetto
× Clienti totali
÷ Clienti per addetto
× Costo per km
```

Esempio:

```text
8.500 clienti
100 clienti per addetto
40.000 km annui per addetto
0,60 €/km
```

Calcolo:

```text
Addetti teorici = 8.500 ÷ 100 = 85
Km totali = 85 × 40.000 = 3.400.000 km
Costo = 3.400.000 × 0,60 € = 2.040.000 €
```

Si definisce teorico perché utilizza `addetti_teorici`, che può essere un numero decimale.

---

## 10.3 `costo_logistico_flotta_addetti_interi`

Formula:

```python
costo_logistico_flotta_addetti_interi = (
    km_totali_con_addetti_interi * costo_km_totale
)
```

Equivalentemente:

```python
costo_logistico_flotta_addetti_interi = (
    km_annui_addetto
    * addetti_necessari_interi
    * costo_km_totale
)
```

Utilizza il numero intero di addetti necessario.

Esempio con 110 clienti per addetto:

```text
8.500 ÷ 110 = 77,27 addetti teorici
```

Il modello arrotonda a:

```text
78 addetti interi
```

Il costo viene quindi calcolato su 78 addetti e non su 77,27.

Questo indicatore è più prudente e generalmente più adatto alla pianificazione del personale e della flotta.

---

## 10.4 `margine_logistico_portafoglio_teorico`

Formula:

```python
margine_logistico_portafoglio_teorico = (
    valore_annuo_portafoglio
    - costo_logistico_flotta_teorica
)
```

Rappresenta il valore complessivo del portafoglio dopo la sottrazione del costo logistico teorico.

Esempio:

```text
Valore portafoglio: 8.500.000 €
Costo logistico teorico: 2.040.000 €
Margine logistico teorico: 6.460.000 €
```

Non è un utile aziendale e non è un margine operativo completo.

È il valore residuo dopo aver sottratto solamente il costo logistico modellizzato.

---

## 10.5 `margine_logistico_portafoglio_addetti_interi`

Formula:

```python
margine_logistico_portafoglio_addetti_interi = (
    valore_annuo_portafoglio
    - costo_logistico_flotta_addetti_interi
)
```

È analogo al margine teorico, ma utilizza il numero intero di addetti necessario.

Questo valore dovrebbe essere preferito quando si vuole formulare una stima operativa prudenziale.

---

## 10.6 `incidenza_costo_logistico_portafoglio_teorico`

Formula:

```python
incidenza_costo_logistico_portafoglio_teorico = (
    costo_logistico_flotta_teorica
    / valore_annuo_portafoglio
)
```

Misura la percentuale del valore annuo complessivo assorbita dalla logistica teorica.

Esempio:

```text
2.040.000 € ÷ 8.500.000 € = 24%
```

---

## 10.7 `incidenza_costo_logistico_portafoglio_addetti_interi`

Formula:

```python
incidenza_costo_logistico_portafoglio_addetti_interi = (
    costo_logistico_flotta_addetti_interi
    / valore_annuo_portafoglio
)
```

Misura la percentuale del valore complessivo assorbita dalla logistica calcolata con il numero intero di addetti.

È generalmente l’indicatore percentuale più prudente.

---

# 11. Parametri effettivamente utilizzati dagli indicatori di flotta

Gli indicatori economici di flotta sono determinati dai seguenti parametri.

| Indicatore                      | Clienti totali | Clienti/addetto | Km/addetto | Costo/km | Valore cliente | Frequenza |
| ------------------------------- | -------------: | --------------: | ---------: | -------: | -------------: | --------: |
| Valore annuo portafoglio        |             Sì |              No |         No |       No |             Sì |        No |
| Km totali flotta teorica        |             Sì |              Sì |         Sì |       No |             No |        No |
| Km totali con addetti interi    |             Sì |              Sì |         Sì |       No |             No |        No |
| Costo flotta teorico            |             Sì |              Sì |         Sì |       Sì |             No |        No |
| Costo flotta con addetti interi |             Sì |              Sì |         Sì |       Sì |             No |        No |
| Margine portafoglio teorico     |             Sì |              Sì |         Sì |       Sì |             Sì |        No |
| Margine con addetti interi      |             Sì |              Sì |         Sì |       Sì |             Sì |        No |
| Incidenza costo portafoglio     |             Sì |              Sì |         Sì |       Sì |             Sì |        No |

La frequenza dei rifornimenti non modifica direttamente gli indicatori annuali di flotta nella formulazione attuale.

Questo avviene perché il modello assume che i chilometri annui per addetto siano già definiti come variabile indipendente.

La frequenza modifica invece:

* rifornimenti mensili;
* rifornimenti giornalieri;
* km attribuiti al singolo rifornimento;
* valore medio per rifornimento;
* costo e margine per rifornimento.

Qualora si ritenga che una maggiore frequenza produca anche un incremento dei chilometri annui, sarà necessario introdurre una relazione esplicita tra:

```text
frequenza dei rifornimenti
```

e:

```text
km annui per addetto
```

Nella versione attuale le due variabili sono indipendenti.

---

# 12. Esempio completo di interpretazione

Si consideri il seguente scenario:

```text
Clienti totali: 8.500
Addetti attuali: 80
Km annui per addetto: 40.000
Clienti per addetto: 100
Frequenza: F1
Valore annuo cliente: 1.000 €
Costo totale per km: 0,60 €
Giorni lavorativi mensili: 22
```

## Risultati operativi

```text
Addetti teorici:
8.500 ÷ 100 = 85

Km mensili per addetto:
40.000 ÷ 12 = 3.333,33

Rifornimenti mensili per addetto:
100 × 1 = 100

Rifornimenti giornalieri:
100 ÷ 22 = 4,55

Km medi per rifornimento:
3.333,33 ÷ 100 = 33,33
```

## Risultati economici unitari

```text
Km annui per cliente:
40.000 ÷ 100 = 400

Costo logistico annuo per cliente:
400 × 0,60 = 240 €

Margine logistico annuo per cliente:
1.000 - 240 = 760 €

Incidenza costo logistico:
240 ÷ 1.000 = 24%
```

## Risultati economici di flotta

```text
Valore annuo portafoglio:
8.500 × 1.000 = 8.500.000 €

Km totali:
85 × 40.000 = 3.400.000 km

Costo logistico totale:
3.400.000 × 0,60 = 2.040.000 €

Margine logistico del portafoglio:
8.500.000 - 2.040.000 = 6.460.000 €

Incidenza del costo logistico:
2.040.000 ÷ 8.500.000 = 24%
```

---

# 13. Come modificare lo scenario economico delle matrici

La funzione:

```python
crea_matrici_standard(...)
```

utilizza per impostazione predefinita:

```python
km_scenario_economico = 40_000
clienti_scenario_economico = 100
frequenza_scenario_economico = "F1"
```

Per creare matrici economiche riferite, per esempio, a:

* 50.000 km annui per addetto;
* 90 clienti per addetto;
* scenario F3;

si deve utilizzare:

```python
matrici = crea_matrici_standard(
    df=df,
    km_scenario_economico=50_000,
    clienti_scenario_economico=90,
    frequenza_scenario_economico="F3",
)
```

In questo caso:

* le righe delle matrici economiche continueranno a rappresentare i diversi costi per km;
* le colonne continueranno a rappresentare i diversi valori annui del cliente;
* tutte le celle saranno calcolate mantenendo fissi 50.000 km, 90 clienti per addetto e frequenza F3.

---

# 14. Avvertenze interpretative

## Il valore cliente è annuale

Tutti i confronti economici assumono che:

```python
valore_annuo_cliente
```

sia espresso su base annua.

Se il valore disponibile è mensile, deve essere convertito:

```text
Valore annuo cliente = valore mensile × 12
```

---

## Il costo per km deve avere un perimetro coerente

Il parametro:

```python
costo_km_totale
```

deve rappresentare sempre lo stesso insieme di costi in tutti gli scenari.

Non bisogna confrontare, per esempio:

* uno scenario con solo carburante;
* un altro scenario con carburante, personale, ammortamento e manutenzione.

---

## Il margine logistico non è il margine aziendale

Gli indicatori denominati “margine logistico” sottraggono soltanto il costo logistico incluso nel modello.

Non includono automaticamente tutti gli altri costi aziendali.

---

## Il valore teorico usa addetti frazionari

Gli indicatori con suffisso:

```text
teorico
```

utilizzano il numero matematico di addetti.

Gli indicatori con riferimento ad:

```text
addetti interi
```

utilizzano il fabbisogno arrotondato all’intero superiore.

---

## I chilometri per cliente e per rifornimento sono valori allocati

Non rappresentano necessariamente la distanza fisica reale.

Sono indicatori medi ottenuti distribuendo i chilometri complessivi tra clienti o rifornimenti.

---

# 15. Sintesi della logica del modello

Il modello segue questa sequenza:

```text
Clienti totali
÷ Clienti per addetto
= Addetti necessari
```

```text
Addetti necessari
× Km annui per addetto
= Km totali della flotta
```

```text
Km totali della flotta
× Costo per km
= Costo logistico complessivo
```

```text
Clienti totali
× Valore annuo cliente
= Valore annuo del portafoglio
```

```text
Valore del portafoglio
- Costo logistico complessivo
= Margine dopo il costo logistico
```

La frequenza dei rifornimenti completa l’analisi operativa:

```text
Clienti per addetto
× Frequenza mensile
= Rifornimenti mensili per addetto
```

```text
Rifornimenti mensili
÷ 22 giorni
= Rifornimenti giornalieri per addetto
```

Questa struttura consente di analizzare contemporaneamente:

* produttività;
* saturazione del personale;
* intensità dei giri;
* sostenibilità del cliente;
* sostenibilità del portafoglio;
* costo della flotta;
* impatto economico delle diverse configurazioni logistiche.


In [ ]:
from pathlib import Path

script = r'''"""
Simulatore scenari logistici ed economici OCS
=============================================

Il modello combina cinque gruppi di variabili:

1. Km annui percorsi per addetto.
2. Clienti serviti per addetto.
3. Frequenza mensile media dei rifornimenti.
4. Valore annuo del cliente.
5. Costo chilometrico totale, comprensivo di addetto e logistica.

Output principali:
- tabella completa di tutte le combinazioni;
- matrici operative a doppia entrata;
- matrici economiche a doppia entrata;
- file Excel con risultati e matrici;
- heatmap opzionali.

Dipendenze:
    pip install pandas numpy openpyxl matplotlib
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd


# =====================================================================
# 1. PARAMETRI MODIFICABILI
# =====================================================================

CLIENTI_OCS = 500
ADDETTI_OCS_ATTUALI = 3
GIORNI_LAVORATIVI_MESE = 22

SCENARI_KM_ANNO_ADDETTO = [
    35_000,
    40_000,
    45_000,
    50_000,
    55_000,
    60_000,
]

SCENARI_CLIENTI_ADDETTO = [
    90,
    100,
    110,
    120,
    130,
    140,
    150,
]

# Assunzione: valore economico ANNUO di un cliente.
# La lista può essere ampliata liberamente.
SCENARI_VALORE_ANNUO_CLIENTE = [
    500,
    750,
    1000,
    1_500,
    2_500,
    5_000,
]

# Costo totale per km: addetto + veicolo + carburante + logistica.
# Un adetto (29.000€) che fa 20.000 km ha un costo per km di 1,75€ con costo di 0,2€ / km per Disel e 0,1€ / km per # # # manutenzione del mezzo. Ogni 10.000 degli scenari aggiuntivi a 20.000 annui, il costo per km si riduce.
# Un adetto che fa 30.000 km il costo per km è di 1,26€. Un adetto che fa 40.000 km il costo per km è di 1,00€.
# Un adetto che fa 50.000 km il costo per km è di 0,88€. Un adetto che fa 60.000 km il costo per km è di 0,78€.
# Un adetto che fa 70.000 km il costo per km è di 0,71€. Un adetto che fa 80.000 km il costo per km è di 0,66€.

SCENARI_COSTO_KM_TOTALE = [
    0.66,
    0.71,
    0.78,
    0.88,
    1.00,
    1.26,
    1.75,
]


@dataclass(frozen=True)
class ScenarioFrequenza:
    codice: str
    descrizione: str
    quota_1: float = 0.0
    quota_2: float = 0.0
    quota_3: float = 0.0
    quota_4: float = 0.0

    @property
    def frequenza_media_mese(self) -> float:
        """Numero medio ponderato di rifornimenti per cliente al mese."""
        return (
            self.quota_1 * 1
            + self.quota_2 * 2
            + self.quota_3 * 3
            + self.quota_4 * 4
        )

    def valida(self) -> None:
        totale_quote = self.quota_1 + self.quota_2 + self.quota_3 + self.quota_4

        if not np.isclose(totale_quote, 1.0):
            raise ValueError(
                f"Le quote dello scenario {self.codice} sommano a "
                f"{totale_quote:.4f}, non a 1."
            )

        if self.frequenza_media_mese <= 0:
            raise ValueError(
                f"La frequenza dello scenario {self.codice} deve essere positiva."
            )


SCENARI_FREQUENZA = [
    ScenarioFrequenza(
        codice="F1",
        descrizione="100% clienti con 1 rifornimento/mese",
        quota_1=1.00,
    ),
    ScenarioFrequenza(
        codice="F2",
        descrizione="50% con 1 e 50% con 2 rifornimenti/mese",
        quota_1=0.50,
        quota_2=0.50,
    ),
    ScenarioFrequenza(
        codice="F3",
        descrizione="50% con 1, 25% con 2 e 25% con 3 rifornimenti/mese",
        quota_1=0.50,
        quota_2=0.25,
        quota_3=0.25,
    ),
    ScenarioFrequenza(
        codice="F4",
        descrizione=(
            "50% con 1, 20% con 2, 20% con 3 e 10% con 4 rifornimenti/mese"
        ),
        quota_1=0.50,
        quota_2=0.20,
        quota_3=0.20,
        quota_4=0.10,
    ),
    ScenarioFrequenza(
        codice="F5",
        descrizione="100% clienti con 2 rifornimenti/mese",
        quota_2=1.00,
    ),
]


# =====================================================================
# 2. CALCOLO DI UNA SINGOLA COMBINAZIONE
# =====================================================================

def calcola_scenario(
    km_annui_addetto: float,
    clienti_addetto: float,
    frequenza_media_mese: float,
    valore_annuo_cliente: float,
    costo_km_totale: float,
    clienti_totali: int = CLIENTI_OCS,
    addetti_attuali: int = ADDETTI_OCS_ATTUALI,
    giorni_lavorativi_mese: int = GIORNI_LAVORATIVI_MESE,
) -> dict:
    """
    Calcola gli indicatori operativi ed economici di una combinazione.

    Interpretazioni:
    - valore_annuo_cliente: ricavo o valore economico annuo medio del cliente;
    - costo_km_totale: costo complessivo per km, comprensivo di addetto e logistica;
    - km_per_rifornimento: quota media di km attribuita a un rifornimento.
    """
    parametri_positivi = {
        "km_annui_addetto": km_annui_addetto,
        "clienti_addetto": clienti_addetto,
        "frequenza_media_mese": frequenza_media_mese,
        "valore_annuo_cliente": valore_annuo_cliente,
        "costo_km_totale": costo_km_totale,
        "giorni_lavorativi_mese": giorni_lavorativi_mese,
    }

    for nome, valore in parametri_positivi.items():
        if valore <= 0:
            raise ValueError(f"{nome} deve essere positivo.")

    # -----------------------------
    # Indicatori operativi
    # -----------------------------
    km_mese_addetto = km_annui_addetto / 12
    km_giorno_addetto = km_mese_addetto / giorni_lavorativi_mese

    km_anno_cliente = km_annui_addetto / clienti_addetto
    km_mese_cliente = km_anno_cliente / 12

    rifornimenti_mese_addetto = clienti_addetto * frequenza_media_mese
    rifornimenti_giorno_addetto = (
        rifornimenti_mese_addetto / giorni_lavorativi_mese
    )
    rifornimenti_anno_addetto = rifornimenti_mese_addetto * 12

    km_per_rifornimento = km_mese_addetto / rifornimenti_mese_addetto

    addetti_teorici = clienti_totali / clienti_addetto
    addetti_necessari_interi = int(np.ceil(addetti_teorici))
    differenza_addetti = addetti_necessari_interi - addetti_attuali

    capacita_clienti_con_addetti_attuali = addetti_attuali * clienti_addetto
    clienti_non_coperti = max(
        0,
        clienti_totali - capacita_clienti_con_addetti_attuali,
    )
    capacita_eccedente_clienti = max(
        0,
        capacita_clienti_con_addetti_attuali - clienti_totali,
    )

    km_totali_flotta_teorica = km_annui_addetto * addetti_teorici
    km_totali_con_addetti_interi = (
        km_annui_addetto * addetti_necessari_interi
    )

    # -----------------------------
    # Indicatori economici unitari
    # -----------------------------
    costo_logistico_annuo_addetto = km_annui_addetto * costo_km_totale
    costo_logistico_mese_addetto = costo_logistico_annuo_addetto / 12

    costo_logistico_annuo_cliente = km_anno_cliente * costo_km_totale
    costo_logistico_mese_cliente = costo_logistico_annuo_cliente / 12

    costo_logistico_per_rifornimento = (
        km_per_rifornimento * costo_km_totale
    )

    valore_annuo_per_addetto = clienti_addetto * valore_annuo_cliente
    valore_mese_per_addetto = valore_annuo_per_addetto / 12

    margine_logistico_annuo_cliente = (
        valore_annuo_cliente - costo_logistico_annuo_cliente
    )
    margine_logistico_annuo_addetto = (
        valore_annuo_per_addetto - costo_logistico_annuo_addetto
    )

    incidenza_costo_logistico_cliente = (
        costo_logistico_annuo_cliente / valore_annuo_cliente
    )
    margine_logistico_percentuale_cliente = (
        margine_logistico_annuo_cliente / valore_annuo_cliente
    )

    valore_medio_per_rifornimento = (
        valore_annuo_cliente / (frequenza_media_mese * 12)
    )
    margine_logistico_per_rifornimento = (
        valore_medio_per_rifornimento - costo_logistico_per_rifornimento
    )

    # -----------------------------
    # Indicatori economici di flotta
    # -----------------------------
    valore_annuo_portafoglio = clienti_totali * valore_annuo_cliente

    costo_logistico_flotta_teorica = (
        km_totali_flotta_teorica * costo_km_totale
    )
    costo_logistico_flotta_addetti_interi = (
        km_totali_con_addetti_interi * costo_km_totale
    )

    margine_logistico_portafoglio_teorico = (
        valore_annuo_portafoglio - costo_logistico_flotta_teorica
    )
    margine_logistico_portafoglio_addetti_interi = (
        valore_annuo_portafoglio - costo_logistico_flotta_addetti_interi
    )

    incidenza_costo_logistico_portafoglio_teorico = (
        costo_logistico_flotta_teorica / valore_annuo_portafoglio
    )
    incidenza_costo_logistico_portafoglio_addetti_interi = (
        costo_logistico_flotta_addetti_interi / valore_annuo_portafoglio
    )

    return {
        # Input
        "km_annui_addetto": km_annui_addetto,
        "clienti_addetto": clienti_addetto,
        "frequenza_media_mese": frequenza_media_mese,
        "valore_annuo_cliente": valore_annuo_cliente,
        "costo_km_totale": costo_km_totale,

        # Operatività
        "km_mese_addetto": km_mese_addetto,
        "km_giorno_addetto": km_giorno_addetto,
        "km_anno_cliente": km_anno_cliente,
        "km_mese_cliente": km_mese_cliente,
        "rifornimenti_mese_addetto": rifornimenti_mese_addetto,
        "rifornimenti_giorno_addetto": rifornimenti_giorno_addetto,
        "rifornimenti_anno_addetto": rifornimenti_anno_addetto,
        "km_per_rifornimento": km_per_rifornimento,

        # Organico
        "addetti_teorici": addetti_teorici,
        "addetti_necessari_interi": addetti_necessari_interi,
        "differenza_vs_addetti_attuali": differenza_addetti,
        "capacita_clienti_con_addetti_attuali": (
            capacita_clienti_con_addetti_attuali
        ),
        "clienti_non_coperti": clienti_non_coperti,
        "capacita_eccedente_clienti": capacita_eccedente_clienti,

        # Km di flotta
        "km_totali_flotta_teorica": km_totali_flotta_teorica,
        "km_totali_con_addetti_interi": km_totali_con_addetti_interi,

        # Economia unitaria
        "costo_logistico_annuo_addetto": costo_logistico_annuo_addetto,
        "costo_logistico_mese_addetto": costo_logistico_mese_addetto,
        "costo_logistico_annuo_cliente": costo_logistico_annuo_cliente,
        "costo_logistico_mese_cliente": costo_logistico_mese_cliente,
        "costo_logistico_per_rifornimento": (
            costo_logistico_per_rifornimento
        ),
        "valore_annuo_per_addetto": valore_annuo_per_addetto,
        "valore_mese_per_addetto": valore_mese_per_addetto,
        "margine_logistico_annuo_cliente": (
            margine_logistico_annuo_cliente
        ),
        "margine_logistico_annuo_addetto": (
            margine_logistico_annuo_addetto
        ),
        "incidenza_costo_logistico_cliente": (
            incidenza_costo_logistico_cliente
        ),
        "margine_logistico_percentuale_cliente": (
            margine_logistico_percentuale_cliente
        ),
        "valore_medio_per_rifornimento": valore_medio_per_rifornimento,
        "margine_logistico_per_rifornimento": (
            margine_logistico_per_rifornimento
        ),

        # Economia di portafoglio
        "valore_annuo_portafoglio": valore_annuo_portafoglio,
        "costo_logistico_flotta_teorica": (
            costo_logistico_flotta_teorica
        ),
        "costo_logistico_flotta_addetti_interi": (
            costo_logistico_flotta_addetti_interi
        ),
        "margine_logistico_portafoglio_teorico": (
            margine_logistico_portafoglio_teorico
        ),
        "margine_logistico_portafoglio_addetti_interi": (
            margine_logistico_portafoglio_addetti_interi
        ),
        "incidenza_costo_logistico_portafoglio_teorico": (
            incidenza_costo_logistico_portafoglio_teorico
        ),
        "incidenza_costo_logistico_portafoglio_addetti_interi": (
            incidenza_costo_logistico_portafoglio_addetti_interi
        ),
    }


# =====================================================================
# 3. GENERAZIONE DI TUTTE LE COMBINAZIONI
# =====================================================================

def genera_tutti_scenari() -> pd.DataFrame:
    righe = []

    for frequenza in SCENARI_FREQUENZA:
        frequenza.valida()

        for km_annui in SCENARI_KM_ANNO_ADDETTO:
            for clienti_addetto in SCENARI_CLIENTI_ADDETTO:
                for valore_cliente in SCENARI_VALORE_ANNUO_CLIENTE:
                    for costo_km in SCENARI_COSTO_KM_TOTALE:
                        risultato = calcola_scenario(
                            km_annui_addetto=km_annui,
                            clienti_addetto=clienti_addetto,
                            frequenza_media_mese=(
                                frequenza.frequenza_media_mese
                            ),
                            valore_annuo_cliente=valore_cliente,
                            costo_km_totale=costo_km,
                        )

                        risultato.update(
                            {
                                "scenario_frequenza": frequenza.codice,
                                "descrizione_frequenza": (
                                    frequenza.descrizione
                                ),
                            }
                        )
                        righe.append(risultato)

    df = pd.DataFrame(righe)

    colonne_iniziali = [
        "scenario_frequenza",
        "descrizione_frequenza",
        "frequenza_media_mese",
        "km_annui_addetto",
        "clienti_addetto",
        "valore_annuo_cliente",
        "costo_km_totale",
    ]

    altre_colonne = [
        colonna
        for colonna in df.columns
        if colonna not in colonne_iniziali
    ]

    return df[colonne_iniziali + altre_colonne].sort_values(
        [
            "scenario_frequenza",
            "km_annui_addetto",
            "clienti_addetto",
            "valore_annuo_cliente",
            "costo_km_totale",
        ]
    )


# =====================================================================
# 4. MATRICI OPERATIVE A DOPPIA ENTRATA
# =====================================================================

def crea_matrice_operativa(
    df: pd.DataFrame,
    variabile_risultato: str,
    scenario_frequenza: str = "F1",
    valore_annuo_cliente: float | None = None,
    costo_km_totale: float | None = None,
) -> pd.DataFrame:
    """
    Matrice operativa:
    - righe: km annui per addetto;
    - colonne: clienti per addetto;
    - celle: indicatore scelto.

    Per evitare duplicazioni derivanti dalle variabili economiche,
    vengono selezionati valori economici specifici.
    """
    if valore_annuo_cliente is None:
        valore_annuo_cliente = SCENARI_VALORE_ANNUO_CLIENTE[0]

    if costo_km_totale is None:
        costo_km_totale = SCENARI_COSTO_KM_TOTALE[0]

    dati = df[
        (df["scenario_frequenza"] == scenario_frequenza)
        & (df["valore_annuo_cliente"] == valore_annuo_cliente)
        & (df["costo_km_totale"] == costo_km_totale)
    ]

    matrice = dati.pivot(
        index="km_annui_addetto",
        columns="clienti_addetto",
        values=variabile_risultato,
    )

    matrice.index.name = "Km annui per addetto"
    matrice.columns.name = "Clienti per addetto"

    return matrice


# =====================================================================
# 5. MATRICI ECONOMICHE A DOPPIA ENTRATA
# =====================================================================

def crea_matrice_economica(
    df: pd.DataFrame,
    variabile_risultato: str,
    km_annui_addetto: float,
    clienti_addetto: float,
    scenario_frequenza: str,
) -> pd.DataFrame:
    """
    Matrice economica:
    - righe: costo totale per km;
    - colonne: valore annuo del cliente;
    - celle: indicatore economico scelto.

    La matrice viene costruita per uno specifico scenario operativo.
    """
    dati = df[
        (df["km_annui_addetto"] == km_annui_addetto)
        & (df["clienti_addetto"] == clienti_addetto)
        & (df["scenario_frequenza"] == scenario_frequenza)
    ]

    matrice = dati.pivot(
        index="costo_km_totale",
        columns="valore_annuo_cliente",
        values=variabile_risultato,
    )

    matrice.index.name = "Costo totale per km (€)"
    matrice.columns.name = "Valore annuo cliente (€)"

    return matrice


def crea_matrici_standard(
    df: pd.DataFrame,
    km_scenario_economico: float = 40_000,
    clienti_scenario_economico: float = 100,
    frequenza_scenario_economico: str = "F1",
) -> Dict[str, pd.DataFrame]:
    """
    Genera il set standard di matrici operative ed economiche.
    """
    matrici: Dict[str, pd.DataFrame] = {}

    # Matrici operative.
    indicatori_operativi = {
        "km_annui_cliente": "km_anno_cliente",
        "km_mensili_cliente": "km_mese_cliente",
        "rifornimenti_mese_addetto": "rifornimenti_mese_addetto",
        "rifornimenti_giorno_addetto": "rifornimenti_giorno_addetto",
        "km_per_rifornimento": "km_per_rifornimento",
        "addetti_teorici": "addetti_teorici",
        "addetti_interi": "addetti_necessari_interi",
    }

    for frequenza in SCENARI_FREQUENZA:
        for nome, indicatore in indicatori_operativi.items():
            nome_matrice = f"{nome}_{frequenza.codice}"
            matrici[nome_matrice] = crea_matrice_operativa(
                df=df,
                variabile_risultato=indicatore,
                scenario_frequenza=frequenza.codice,
            )

    # Matrici economiche per lo scenario operativo selezionato.
    indicatori_economici = {
        "costo_annuo_cliente": "costo_logistico_annuo_cliente",
        "costo_per_rifornimento": "costo_logistico_per_rifornimento",
        "margine_annuo_cliente": "margine_logistico_annuo_cliente",
        "margine_per_rifornimento": "margine_logistico_per_rifornimento",
        "incidenza_costo_cliente": "incidenza_costo_logistico_cliente",
        "margine_percentuale_cliente": (
            "margine_logistico_percentuale_cliente"
        ),
        "margine_portafoglio": (
            "margine_logistico_portafoglio_addetti_interi"
        ),
        "incidenza_costo_portafoglio": (
            "incidenza_costo_logistico_portafoglio_addetti_interi"
        ),
    }

    for nome, indicatore in indicatori_economici.items():
        matrici[nome] = crea_matrice_economica(
            df=df,
            variabile_risultato=indicatore,
            km_annui_addetto=km_scenario_economico,
            clienti_addetto=clienti_scenario_economico,
            scenario_frequenza=frequenza_scenario_economico,
        )

    return matrici


# =====================================================================
# 6. SELEZIONE DI UNO SCENARIO SPECIFICO
# =====================================================================

def seleziona_scenario(
    df: pd.DataFrame,
    km_annui_addetto: float,
    clienti_addetto: float,
    scenario_frequenza: str,
    valore_annuo_cliente: float,
    costo_km_totale: float,
) -> pd.Series:
    filtro = (
        (df["km_annui_addetto"] == km_annui_addetto)
        & (df["clienti_addetto"] == clienti_addetto)
        & (df["scenario_frequenza"] == scenario_frequenza)
        & (df["valore_annuo_cliente"] == valore_annuo_cliente)
        & (df["costo_km_totale"] == costo_km_totale)
    )

    risultato = df.loc[filtro]

    if risultato.empty:
        raise KeyError("La combinazione richiesta non esiste.")

    if len(risultato) > 1:
        raise RuntimeError("La combinazione richiesta non è univoca.")

    return risultato.iloc[0]


# =====================================================================
# 7. FUNZIONE PER LA STAMPA DEI PARAMETRI ATTIVI
# =====================================================================

def stampa_parametri_attivi() -> pd.DataFrame:
    """Restituisce un DataFrame con i parametri attivi del simulatore."""
    parametri_df = pd.DataFrame(
        {
            "Parametro": [
                "Clienti OCS",
                "Addetti OCS attuali",
                "Giorni lavorativi mese",
                "Scenari km annui/addetto",
                "Scenari clienti/addetto",
                "Scenari valore annuo cliente",
                "Scenari costo totale/km",
            ],
            "Valore": [
                CLIENTI_OCS,
                ADDETTI_OCS_ATTUALI,
                GIORNI_LAVORATIVI_MESE,
                ", ".join(map(str, SCENARI_KM_ANNO_ADDETTO)),
                ", ".join(map(str, SCENARI_CLIENTI_ADDETTO)),
                ", ".join(map(str, SCENARI_VALORE_ANNUO_CLIENTE)),
                ", ".join(map(str, SCENARI_COSTO_KM_TOTALE)),
            ],
        }
    )
    return parametri_df


# =====================================================================
# 8. ESPORTAZIONE EXCEL
# =====================================================================

def esporta_excel(
    df: pd.DataFrame,
    matrici: Dict[str, pd.DataFrame],
    percorso: str | Path = "risultati_scenari_ocs_economici.xlsx",
) -> Path:
    percorso = Path(percorso)

    frequenze_df = pd.DataFrame(
        [
            {
                "codice": frequenza.codice,
                "descrizione": frequenza.descrizione,
                "quota_1": frequenza.quota_1,
                "quota_2": frequenza.quota_2,
                "quota_3": frequenza.quota_3,
                "quota_4": frequenza.quota_4,
                "frequenza_media_mese": (
                    frequenza.frequenza_media_mese
                ),
            }
            for frequenza in SCENARI_FREQUENZA
        ]
    )

    parametri_df = pd.DataFrame(
        {
            "Parametro": [
                "Clienti OCS",
                "Addetti OCS attuali",
                "Giorni lavorativi mese",
                "Scenari km annui/addetto",
                "Scenari clienti/addetto",
                "Scenari valore annuo cliente",
                "Scenari costo totale/km",
            ],
            "Valore": [
                CLIENTI_OCS,
                ADDETTI_OCS_ATTUALI,
                GIORNI_LAVORATIVI_MESE,
                ", ".join(map(str, SCENARI_KM_ANNO_ADDETTO)),
                ", ".join(map(str, SCENARI_CLIENTI_ADDETTO)),
                ", ".join(map(str, SCENARI_VALORE_ANNUO_CLIENTE)),
                ", ".join(map(str, SCENARI_COSTO_KM_TOTALE)),
            ],
        }
    )

    with pd.ExcelWriter(percorso, engine="openpyxl") as writer:
        df.to_excel(
            writer,
            sheet_name="Tutti_scenari",
            index=False,
        )
        frequenze_df.to_excel(
            writer,
            sheet_name="Frequenze",
            index=False,
        )
        parametri_df.to_excel(
            writer,
            sheet_name="Parametri",
            index=False,
        )

        nomi_usati: set[str] = set()

        for nome, matrice in matrici.items():
            nome_base = nome[:31]
            nome_foglio = nome_base
            contatore = 1

            while nome_foglio in nomi_usati:
                suffisso = f"_{contatore}"
                nome_foglio = nome_base[: 31 - len(suffisso)] + suffisso
                contatore += 1

            nomi_usati.add(nome_foglio)
            matrice.to_excel(writer, sheet_name=nome_foglio)

    return percorso.resolve()


# =====================================================================
# 9. HEATMAP OPZIONALE
# =====================================================================

def salva_heatmap(
    matrice: pd.DataFrame,
    titolo: str,
    percorso: str | Path,
    formato: str = ".1f",
) -> Path:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 6))
    immagine = ax.imshow(matrice.values, aspect="auto")

    ax.set_xticks(range(len(matrice.columns)))
    ax.set_xticklabels(matrice.columns)
    ax.set_yticks(range(len(matrice.index)))
    ax.set_yticklabels(matrice.index)

    ax.set_xlabel(matrice.columns.name or "Colonne")
    ax.set_ylabel(matrice.index.name or "Righe")
    ax.set_title(titolo)

    for riga in range(matrice.shape[0]):
        for colonna in range(matrice.shape[1]):
            valore = matrice.iloc[riga, colonna]
            ax.text(
                colonna,
                riga,
                format(valore, formato),
                ha="center",
                va="center",
            )

    fig.colorbar(immagine, ax=ax)
    fig.tight_layout()

    percorso = Path(percorso)
    fig.savefig(percorso, dpi=180, bbox_inches="tight")
    plt.close(fig)

    return percorso.resolve()


# =====================================================================
# 10. ESECUZIONE
# =====================================================================

def main() -> None:
    df = genera_tutti_scenari()

    matrici = crea_matrici_standard(
        df=df,
        km_scenario_economico=50_000,
        clienti_scenario_economico=150,
        frequenza_scenario_economico="F1",
    )

    numero_combinazioni_attese = (
        len(SCENARI_KM_ANNO_ADDETTO)
        * len(SCENARI_CLIENTI_ADDETTO)
        * len(SCENARI_FREQUENZA)
        * len(SCENARI_VALORE_ANNUO_CLIENTE)
        * len(SCENARI_COSTO_KM_TOTALE)
    )

    print(f"Combinazioni generate: {len(df):,}")
    print(f"Combinazioni attese:   {numero_combinazioni_attese:,}")
    print()

    print("PARAMETRI ATTIVI")
    print(stampa_parametri_attivi().to_string(index=False))
    print()

    # Parametri utilizzati per lo "SCENARIO DI ESEMPIO":
    # - km_annui_addetto: 50_000
    # - clienti_addetto: 150
    # - scenario_frequenza: "F1"
    # - valore_annuo_cliente: 1_000
    # - costo_km_totale: 0.88

    # SELEZIONE DELLA VARIABILE costo_km_totale:
    # Un adetto che fa 30.000 km il costo per km è di 1,26€.
    # Un adetto che fa 40.000 km il costo per km è di 1,00€.
    # Un adetto che fa 50.000 km il costo per km è di 0,88€.
    # Un adetto che fa 60.000 km il costo per km è di 0,78€.
    # Un adetto che fa 70.000 km il costo per km è di 0,71€.
    # Un adetto che fa 80.000 km il costo per km è di 0,66€.

    scenario_esempio = seleziona_scenario(
        df=df,
        km_annui_addetto=50_000,
        clienti_addetto=150,
        scenario_frequenza="F1",
        valore_annuo_cliente=1_000,
        costo_km_totale=0.88,
    )

    indicatori_esempio = [
        "km_mese_addetto",
        "km_giorno_addetto",
        "rifornimenti_mese_addetto",
        "rifornimenti_giorno_addetto",
        "km_per_rifornimento",
        "costo_logistico_annuo_cliente",
        "costo_logistico_per_rifornimento",
        "margine_logistico_annuo_cliente",
        "incidenza_costo_logistico_cliente",
        "addetti_necessari_interi",
        "costo_logistico_flotta_addetti_interi",
        "margine_logistico_portafoglio_addetti_interi",
    ]

    print("SCENARIO DI ESEMPIO")
    print(scenario_esempio[indicatori_esempio].to_string())
    print()

    print("MATRICE: RIFORNIMENTI GIORNALIERI PER ADDETTO – F1")
    print(matrici["rifornimenti_giorno_addetto_F1"].round(2))
    print()

    print("MATRICE: KM PER RIFORNIMENTO – F1")
    print(matrici["km_per_rifornimento_F1"].round(2))
    print()

    print("MATRICE ECONOMICA: INCIDENZA COSTO LOGISTICO SUL CLIENTE")
    print((matrici["incidenza_costo_cliente"] * 100).round(2))
    print()

    file_excel = esporta_excel(
        df=df,
        matrici=matrici,
        percorso="risultati_scenari_ocs_economici.xlsx",
    )

    print(f"File Excel creato: {file_excel}")

    salva_heatmap(
        matrice=matrici["incidenza_costo_cliente"] * 100,
        titolo=(
            "Incidenza % del costo logistico sul valore cliente\n"
            "40.000 km/addetto – 100 clienti/addetto – F1"
        ),
        percorso="heatmap_incidenza_costo_cliente.png",
        formato=".1f",
    )


if __name__ == "__main__":
    main()
'''

path = Path("/mnt/data/simulatore_scenari_ocs_economici.py")
# Create the parent directory if it doesn't exist
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(script, encoding="utf-8")
print(path)
print(f"Righe: {len(script.splitlines())}")

/mnt/data/simulatore_scenari_ocs_economici.py
Righe: 879


In [ ]:
import subprocess
import sys
import os

# Esegui il nuovo script per generare i risultati economici
result = subprocess.run(
    [sys.executable, "/mnt/data/simulatore_scenari_ocs_economici.py"],
    cwd="/mnt/data",
    capture_output=True,
    text=True,
    timeout=240, # Aumentato il timeout per gestire più combinazioni
)

print(result.stdout)
print(result.stderr)
print("returncode:", result.returncode)


Combinazioni generate: 8,820
Combinazioni attese:   8,820

PARAMETRI ATTIVI
                   Parametro                                   Valore
                 Clienti OCS                                      500
         Addetti OCS attuali                                        3
      Giorni lavorativi mese                                       22
    Scenari km annui/addetto 35000, 40000, 45000, 50000, 55000, 60000
     Scenari clienti/addetto         90, 100, 110, 120, 130, 140, 150
Scenari valore annuo cliente         500, 750, 1000, 1500, 2500, 5000
     Scenari costo totale/km  0.66, 0.71, 0.78, 0.88, 1.0, 1.26, 1.75

SCENARIO DI ESEMPIO
km_mese_addetto                                 4166.666667
km_giorno_addetto                                189.393939
rifornimenti_mese_addetto                             150.0
rifornimenti_giorno_addetto                        6.818182
km_per_rifornimento                               27.777778
costo_logistico_annuo_cliente              

---

In [ ]:
import pandas as pd
from IPython.display import display

excel_path = "/mnt/data/risultati_scenari_ocs_economici.xlsx"

# Carica il file Excel
xl = pd.ExcelFile(excel_path)

print(f"Schede trovate nel file Excel '{excel_path}':")
for sheet_name in xl.sheet_names:
    print(f"- {sheet_name}")

# Esempio: Mostra le prime righe della scheda 'Tutti_scenari' in formato verticale
print("\n--- Prime 5  colonne della scheda 'Tutti_scenari' (formato verticale) ---")
df_scenari = xl.parse("Tutti_scenari")
display(df_scenari.head().T) # Transpose for vertical display

# Esempio: Mostra la matrice di incidenza del costo sul cliente in formato verticale
print("\n--- Matrice 'incidenza_costo_cliente' (primi valori, formato verticale) ---")
df_incidenza = xl.parse("incidenza_costo_cliente")
display(df_incidenza.head().T) # Transpose for vertical display

# Salva le schede in file CSV separati
df_scenari.to_csv('/mnt/data/tutti_scenari.csv', index=False)
df_incidenza.to_csv('/mnt/data/incidenza_costo_cliente.csv', index=False)
print("\nLe schede 'Tutti_scenari' e 'incidenza_costo_cliente' sono state salvate come file CSV in /mnt/data/")

Schede trovate nel file Excel '/mnt/data/risultati_scenari_ocs_economici.xlsx':
- Tutti_scenari
- Frequenze
- Parametri
- km_annui_cliente_F1
- km_mensili_cliente_F1
- rifornimenti_mese_addetto_F1
- rifornimenti_giorno_addetto_F1
- km_per_rifornimento_F1
- addetti_teorici_F1
- addetti_interi_F1
- km_annui_cliente_F2
- km_mensili_cliente_F2
- rifornimenti_mese_addetto_F2
- rifornimenti_giorno_addetto_F2
- km_per_rifornimento_F2
- addetti_teorici_F2
- addetti_interi_F2
- km_annui_cliente_F3
- km_mensili_cliente_F3
- rifornimenti_mese_addetto_F3
- rifornimenti_giorno_addetto_F3
- km_per_rifornimento_F3
- addetti_teorici_F3
- addetti_interi_F3
- km_annui_cliente_F4
- km_mensili_cliente_F4
- rifornimenti_mese_addetto_F4
- rifornimenti_giorno_addetto_F4
- km_per_rifornimento_F4
- addetti_teorici_F4
- addetti_interi_F4
- km_annui_cliente_F5
- km_mensili_cliente_F5
- rifornimenti_mese_addetto_F5
- rifornimenti_giorno_addetto_F5
- km_per_rifornimento_F5
- addetti_teorici_F5
- addetti_interi_F5


,0,1,2,3,4
scenario_frequenza,F1,F1,F1,F1,F1
descrizione_frequenza,100% clienti con 1 rifornimento/mese,100% clienti con 1 rifornimento/mese,100% clienti con 1 rifornimento/mese,100% clienti con 1 rifornimento/mese,100% clienti con 1 rifornimento/mese
frequenza_media_mese,1.0,1.0,1.0,1.0,1.0
km_annui_addetto,35000,35000,35000,35000,35000
clienti_addetto,90,90,90,90,90
valore_annuo_cliente,500,500,500,500,500
costo_km_totale,0.66,0.71,0.78,0.88,1.0
km_mese_addetto,2916.666667,2916.666667,2916.666667,2916.666667,2916.666667
km_giorno_addetto,132.575758,132.575758,132.575758,132.575758,132.575758
km_anno_cliente,388.888889,388.888889,388.888889,388.888889,388.888889



--- Matrice 'incidenza_costo_cliente' (primi valori, formato verticale) ---


,0,1,2,3,4
Costo totale per km (€),0.660000,0.710000,0.780000,0.880000,1.000000
500,0.440000,0.473333,0.520000,0.586667,0.666667
750,0.293333,0.315556,0.346667,0.391111,0.444444
1000,0.220000,0.236667,0.260000,0.293333,0.333333
1500,0.146667,0.157778,0.173333,0.195556,0.222222
2500,0.088000,0.094667,0.104000,0.117333,0.133333
5000,0.044000,0.047333,0.052000,0.058667,0.066667



Le schede 'Tutti_scenari' e 'incidenza_costo_cliente' sono state salvate come file CSV in /mnt/data/


In [ ]:
from pathlib import Path

code = r'''"""
SIMULATORE OCS REGIONALE — FLOTTA REALE E BENCHMARK MULTIPLI
=============================================================

Logica del modello
------------------
Il modello separa due livelli che non devono essere confusi:

A) AREA REGIONALE REALE
   È descritta dai dati effettivamente inseriti dal manager:
   - clienti reali dell'area;
   - addetti reali dell'area;
   - km annui medi per addetto;
   - frequenza dei rifornimenti;
   - valore annuo medio del cliente;
   - costo totale per km.

   Tutti i risultati economici principali della regione sono calcolati
   sugli ADDETTI REALI, non sugli addetti teorici.

B) BENCHMARK / SCENARIO DI CONFRONTO
   Il parametro "clienti benchmark per addetto" rappresenta la produttività
   di un'altra regione o una configurazione-obiettivo.

   Serve per calcolare:
   - addetti teorici richiesti dal benchmark;
   - addetti interi richiesti;
   - scostamento rispetto agli addetti reali;
   - capacità teorica del benchmark;
   - costi e margini controfattuali se l'area adottasse quel benchmark.

Il report dello scenario scelto pubblica separatamente:
1. dati reali dell'area;
2. risultati reali;
3. risultati benchmark;
4. scostamenti reale vs benchmark.

Dipendenze:
    pip install pandas numpy openpyxl
"""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict

import numpy as np
import pandas as pd


# =====================================================================
# 1. INPUT DELL'AREA REGIONALE REALE
# =====================================================================

NOME_AREA = "Marche"

CLIENTI_OCS_REALI = 600
ADDETTI_OCS_REALI = 3
GIORNI_LAVORATIVI_MESE = 22

# Gli scenari rappresentano parametri reali o benchmark di regioni diverse.
SCENARI_KM_ANNO_ADDETTO = [35_000, 40_000, 45_000, 50_000, 55_000, 60_000]
SCENARI_CLIENTI_BENCHMARK_ADDETTO = [85, 90, 95, 100, 105, 110, 150]
SCENARI_VALORE_ANNUO_CLIENTE = [100, 200, 500, 1_000, 2_000, 5_000]
SCENARI_COSTO_KM_TOTALE = [0.30, 0.40, 0.50, 0.60, 0.88, 1.00, 1.50, 2.00]


@dataclass(frozen=True)
class ScenarioFrequenza:
    codice: str
    descrizione: str
    quota_1: float = 0.0
    quota_2: float = 0.0
    quota_3: float = 0.0
    quota_4: float = 0.0

    @property
    def frequenza_media_mese(self) -> float:
        return (
            self.quota_1
            + 2 * self.quota_2
            + 3 * self.quota_3
            + 4 * self.quota_4
        )

    def valida(self) -> None:
        totale = self.quota_1 + self.quota_2 + self.quota_3 + self.quota_4
        if not np.isclose(totale, 1.0):
            raise ValueError(
                f"Le quote dello scenario {self.codice} sommano a "
                f"{totale:.4f}, non a 1."
            )
        if self.frequenza_media_mese <= 0:
            raise ValueError("La frequenza media deve essere positiva.")


SCENARI_FREQUENZA = [
    ScenarioFrequenza(
        "F1",
        "100% clienti con 1 rifornimento/mese",
        quota_1=1.00,
    ),
    ScenarioFrequenza(
        "F2",
        "50% con 1 e 50% con 2 rifornimenti/mese",
        quota_1=0.50,
        quota_2=0.50,
    ),
    ScenarioFrequenza(
        "F3",
        "50% con 1, 25% con 2 e 25% con 3 rifornimenti/mese",
        quota_1=0.50,
        quota_2=0.25,
        quota_3=0.25,
    ),
    ScenarioFrequenza(
        "F4",
        "50% con 1, 20% con 2, 20% con 3 e 10% con 4 rifornimenti/mese",
        quota_1=0.50,
        quota_2=0.20,
        quota_3=0.20,
        quota_4=0.10,
    ),
    ScenarioFrequenza(
        "F5",
        "100% clienti con 2 rifornimenti/mese",
        quota_2=1.00,
    ),
]


# =====================================================================
# 2. CALCOLO DI UNO SCENARIO REGIONALE
# =====================================================================

def calcola_scenario_regionale(
    km_annui_addetto: float,
    clienti_benchmark_addetto: float,
    frequenza_media_mese: float,
    valore_annuo_cliente: float,
    costo_km_totale: float,
    clienti_reali: int = CLIENTI_OCS_REALI,
    addetti_reali: int = ADDETTI_OCS_REALI,
    giorni_lavorativi_mese: int = GIORNI_LAVORATIVI_MESE,
) -> dict:
    """
    Calcola contemporaneamente:

    1. risultati REALI dell'area, usando clienti_reali e addetti_reali;
    2. risultati BENCHMARK, usando clienti_benchmark_addetto;
    3. scostamenti tra struttura reale e benchmark.

    Il parametro km_annui_addetto è applicato sia alla flotta reale sia al
    controfattuale benchmark. In questo modo si isola l'effetto del diverso
    rapporto clienti/addetto.

    Per confrontare regioni con chilometraggi diversi, km_annui_addetto viene
    variato nella simulazione multipla.
    """

    parametri_positivi = {
        "km_annui_addetto": km_annui_addetto,
        "clienti_benchmark_addetto": clienti_benchmark_addetto,
        "frequenza_media_mese": frequenza_media_mese,
        "valore_annuo_cliente": valore_annuo_cliente,
        "costo_km_totale": costo_km_totale,
        "clienti_reali": clienti_reali,
        "addetti_reali": addetti_reali,
        "giorni_lavorativi_mese": giorni_lavorativi_mese,
    }
    for nome, valore in parametri_positivi.items():
        if valore <= 0:
            raise ValueError(f"{nome} deve essere positivo.")

    # -----------------------------------------------------------------
    # A. STRUTTURA REALE DELL'AREA
    # -----------------------------------------------------------------
    clienti_reali_per_addetto = clienti_reali / addetti_reali
    valore_annuo_totale_clienti = clienti_reali * valore_annuo_cliente
    valore_annuo_reale_per_addetto = (
        valore_annuo_totale_clienti / addetti_reali
    )

    km_annui_flotta_reale = km_annui_addetto * addetti_reali
    km_mese_flotta_reale = km_annui_flotta_reale / 12

    km_mese_reale_per_addetto = km_annui_addetto / 12
    km_giorno_reale_per_addetto = (
        km_mese_reale_per_addetto / giorni_lavorativi_mese
    )

    km_annui_reali_per_cliente = km_annui_flotta_reale / clienti_reali
    km_mese_reali_per_cliente = km_annui_reali_per_cliente / 12

    rifornimenti_mese_area_reale = clienti_reali * frequenza_media_mese
    rifornimenti_anno_area_reale = rifornimenti_mese_area_reale * 12

    rifornimenti_mese_reali_per_addetto = (
        rifornimenti_mese_area_reale / addetti_reali
    )
    rifornimenti_giorno_reali_per_addetto = (
        rifornimenti_mese_reali_per_addetto / giorni_lavorativi_mese
    )

    km_reali_per_rifornimento = (
        km_mese_flotta_reale / rifornimenti_mese_area_reale
    )

    costo_logistico_annuo_flotta_reale = (
        km_annui_flotta_reale * costo_km_totale
    )
    costo_logistico_annuo_reale_per_addetto = (
        costo_logistico_annuo_flotta_reale / addetti_reali
    )
    costo_logistico_annuo_reale_per_cliente = (
        costo_logistico_annuo_flotta_reale / clienti_reali
    )
    costo_logistico_reale_per_rifornimento = (
        costo_logistico_annuo_flotta_reale / rifornimenti_anno_area_reale
    )

    margine_logistico_annuo_area_reale = (
        valore_annuo_totale_clienti - costo_logistico_annuo_flotta_reale
    )
    margine_logistico_annuo_reale_per_addetto = (
        margine_logistico_annuo_area_reale / addetti_reali
    )
    margine_logistico_annuo_reale_per_cliente = (
        margine_logistico_annuo_area_reale / clienti_reali
    )
    incidenza_costo_logistico_area_reale = (
        costo_logistico_annuo_flotta_reale / valore_annuo_totale_clienti
    )

    # -----------------------------------------------------------------
    # B. BENCHMARK / CONTROFATTUALE
    # -----------------------------------------------------------------
    addetti_benchmark_teorici = (
        clienti_reali / clienti_benchmark_addetto
    )
    addetti_benchmark_interi = int(np.ceil(addetti_benchmark_teorici))

    differenza_addetti_benchmark_vs_reali = (
        addetti_benchmark_interi - addetti_reali
    )

    capacita_clienti_benchmark_con_addetti_reali = (
        clienti_benchmark_addetto * addetti_reali
    )
    clienti_non_coperti_al_benchmark = max(
        0,
        clienti_reali - capacita_clienti_benchmark_con_addetti_reali,
    )
    capacita_eccedente_al_benchmark = max(
        0,
        capacita_clienti_benchmark_con_addetti_reali - clienti_reali,
    )

    # Benchmark teorico continuo: ammette frazioni di addetto.
    km_annui_flotta_benchmark_teorica = (
        km_annui_addetto * addetti_benchmark_teorici
    )
    costo_logistico_benchmark_teorico = (
        km_annui_flotta_benchmark_teorica * costo_km_totale
    )

    # Benchmark operativo: arrotonda gli addetti all'intero superiore.
    km_annui_flotta_benchmark_interi = (
        km_annui_addetto * addetti_benchmark_interi
    )
    costo_logistico_benchmark_interi = (
        km_annui_flotta_benchmark_interi * costo_km_totale
    )
    margine_logistico_benchmark_interi = (
        valore_annuo_totale_clienti - costo_logistico_benchmark_interi
    )
    incidenza_costo_logistico_benchmark_interi = (
        costo_logistico_benchmark_interi / valore_annuo_totale_clienti
    )

    valore_annuo_benchmark_per_addetto_teorico = (
        clienti_benchmark_addetto * valore_annuo_cliente
    )
    valore_annuo_benchmark_per_addetto_intero = (
        valore_annuo_totale_clienti / addetti_benchmark_interi
    )

    rifornimenti_mese_benchmark_per_addetto = (
        clienti_benchmark_addetto * frequenza_media_mese
    )
    rifornimenti_giorno_benchmark_per_addetto = (
        rifornimenti_mese_benchmark_per_addetto
        / giorni_lavorativi_mese
    )

    # -----------------------------------------------------------------
    # C. SCOSTAMENTI REALE VS BENCHMARK OPERATIVO
    # -----------------------------------------------------------------
    differenza_clienti_per_addetto = (
        clienti_reali_per_addetto - clienti_benchmark_addetto
    )
    differenza_km_flotta_reale_vs_benchmark = (
        km_annui_flotta_reale - km_annui_flotta_benchmark_interi
    )
    differenza_costo_reale_vs_benchmark = (
        costo_logistico_annuo_flotta_reale
        - costo_logistico_benchmark_interi
    )
    differenza_margine_reale_vs_benchmark = (
        margine_logistico_annuo_area_reale
        - margine_logistico_benchmark_interi
    )

    risparmio_potenziale_benchmark = (
        costo_logistico_annuo_flotta_reale
        - costo_logistico_benchmark_interi
    )

    return {
        # Identificazione dell'area e input
        "area": NOME_AREA,
        "clienti_reali_area": clienti_reali,
        "addetti_reali_area": addetti_reali,
        "giorni_lavorativi_mese": giorni_lavorativi_mese,
        "km_annui_addetto": km_annui_addetto,
        "clienti_benchmark_addetto": clienti_benchmark_addetto,
        "frequenza_media_mese": frequenza_media_mese,
        "valore_annuo_cliente": valore_annuo_cliente,
        "costo_km_totale": costo_km_totale,

        # Risultati reali dell'area
        "clienti_reali_per_addetto": clienti_reali_per_addetto,
        "valore_annuo_totale_clienti": valore_annuo_totale_clienti,
        "valore_annuo_reale_per_addetto": valore_annuo_reale_per_addetto,
        "km_annui_flotta_reale": km_annui_flotta_reale,
        "km_mese_flotta_reale": km_mese_flotta_reale,
        "km_mese_reale_per_addetto": km_mese_reale_per_addetto,
        "km_giorno_reale_per_addetto": km_giorno_reale_per_addetto,
        "km_annui_reali_per_cliente": km_annui_reali_per_cliente,
        "km_mese_reali_per_cliente": km_mese_reali_per_cliente,
        "rifornimenti_mese_area_reale": rifornimenti_mese_area_reale,
        "rifornimenti_anno_area_reale": rifornimenti_anno_area_reale,
        "rifornimenti_mese_reali_per_addetto": (
            rifornimenti_mese_reali_per_addetto
        ),
        "rifornimenti_giorno_reali_per_addetto": (
            rifornimenti_giorno_reali_per_addetto
        ),
        "km_reali_per_rifornimento": km_reali_per_rifornimento,
        "costo_logistico_annuo_flotta_reale": (
            costo_logistico_annuo_flotta_reale
        ),
        "costo_logistico_annuo_reale_per_addetto": (
            costo_logistico_annuo_reale_per_addetto
        ),
        "costo_logistico_annuo_reale_per_cliente": (
            costo_logistico_annuo_reale_per_cliente
        ),
        "costo_logistico_reale_per_rifornimento": (
            costo_logistico_reale_per_rifornimento
        ),
        "margine_logistico_annuo_area_reale": (
            margine_logistico_annuo_area_reale
        ),
        "margine_logistico_annuo_reale_per_addetto": (
            margine_logistico_annuo_reale_per_addetto
        ),
        "margine_logistico_annuo_reale_per_cliente": (
            margine_logistico_annuo_reale_per_cliente
        ),
        "incidenza_costo_logistico_area_reale": (
            incidenza_costo_logistico_area_reale
        ),

        # Benchmark
        "addetti_benchmark_teorici": addetti_benchmark_teorici,
        "addetti_benchmark_interi": addetti_benchmark_interi,
        "differenza_addetti_benchmark_vs_reali": (
            differenza_addetti_benchmark_vs_reali
        ),
        "capacita_clienti_benchmark_con_addetti_reali": (
            capacita_clienti_benchmark_con_addetti_reali
        ),
        "clienti_non_coperti_al_benchmark": clienti_non_coperti_al_benchmark,
        "capacita_eccedente_al_benchmark": capacita_eccedente_al_benchmark,
        "km_annui_flotta_benchmark_teorica": (
            km_annui_flotta_benchmark_teorica
        ),
        "costo_logistico_benchmark_teorico": (
            costo_logistico_benchmark_teorico
        ),
        "km_annui_flotta_benchmark_interi": (
            km_annui_flotta_benchmark_interi
        ),
        "costo_logistico_benchmark_interi": (
            costo_logistico_benchmark_interi
        ),
        "margine_logistico_benchmark_interi": (
            margine_logistico_benchmark_interi
        ),
        "incidenza_costo_logistico_benchmark_interi": (
            incidenza_costo_logistico_benchmark_interi
        ),
        "valore_annuo_benchmark_per_addetto_teorico": (
            valore_annuo_benchmark_per_addetto_teorico
        ),
        "valore_annuo_benchmark_per_addetto_intero": (
            valore_annuo_benchmark_per_addetto_intero
        ),
        "rifornimenti_mese_benchmark_per_addetto": (
            rifornimenti_mese_benchmark_per_addetto
        ),
        "rifornimenti_giorno_benchmark_per_addetto": (
            rifornimenti_giorno_benchmark_per_addetto
        ),

        # Scostamenti
        "differenza_clienti_per_addetto_reale_vs_benchmark": (
            differenza_clienti_per_addetto
        ),
        "differenza_km_flotta_reale_vs_benchmark": (
            differenza_km_flotta_reale_vs_benchmark
        ),
        "differenza_costo_reale_vs_benchmark": (
            differenza_costo_reale_vs_benchmark
        ),
        "differenza_margine_reale_vs_benchmark": (
            differenza_margine_reale_vs_benchmark
        ),
        "risparmio_potenziale_benchmark": risparmio_potenziale_benchmark,
    }


# =====================================================================
# 3. GENERAZIONE DI TUTTI GLI SCENARI
# =====================================================================

def genera_tutti_scenari() -> pd.DataFrame:
    righe = []

    for frequenza in SCENARI_FREQUENZA:
        frequenza.valida()

        for km_annui in SCENARI_KM_ANNO_ADDETTO:
            for clienti_benchmark in SCENARI_CLIENTI_BENCHMARK_ADDETTO:
                for valore_cliente in SCENARI_VALORE_ANNUO_CLIENTE:
                    for costo_km in SCENARI_COSTO_KM_TOTALE:
                        risultato = calcola_scenario_regionale(
                            km_annui_addetto=km_annui,
                            clienti_benchmark_addetto=clienti_benchmark,
                            frequenza_media_mese=(
                                frequenza.frequenza_media_mese
                            ),
                            valore_annuo_cliente=valore_cliente,
                            costo_km_totale=costo_km,
                        )
                        risultato["scenario_frequenza"] = frequenza.codice
                        risultato["descrizione_frequenza"] = (
                            frequenza.descrizione
                        )
                        righe.append(risultato)

    df = pd.DataFrame(righe)

    ordine = [
        "area",
        "scenario_frequenza",
        "descrizione_frequenza",
        "clienti_reali_area",
        "addetti_reali_area",
        "km_annui_addetto",
        "clienti_benchmark_addetto",
        "frequenza_media_mese",
        "valore_annuo_cliente",
        "costo_km_totale",
    ]

    altre = [colonna for colonna in df.columns if colonna not in ordine]
    return df[ordine + altre].sort_values(
        [
            "scenario_frequenza",
            "km_annui_addetto",
            "clienti_benchmark_addetto",
            "valore_annuo_cliente",
            "costo_km_totale",
        ]
    )


# =====================================================================
# 4. SELEZIONE DELLO SCENARIO DA PUBBLICARE
# =====================================================================

def seleziona_scenario(
    df: pd.DataFrame,
    km_annui_addetto: float,
    clienti_benchmark_addetto: float,
    scenario_frequenza: str,
    valore_annuo_cliente: float,
    costo_km_totale: float,
) -> pd.Series:
    filtro = (
        (df["km_annui_addetto"] == km_annui_addetto)
        & (
            df["clienti_benchmark_addetto"]
            == clienti_benchmark_addetto
        )
        & (df["scenario_frequenza"] == scenario_frequenza)
        & (df["valore_annuo_cliente"] == valore_annuo_cliente)
        & (df["costo_km_totale"] == costo_km_totale)
    )

    risultato = df.loc[filtro]

    if risultato.empty:
        raise KeyError("La combinazione richiesta non esiste.")
    if len(risultato) > 1:
        raise RuntimeError("La combinazione richiesta non è univoca.")

    return risultato.iloc[0]


def crea_report_scenario(scenario: pd.Series) -> pd.DataFrame:
    """
    Crea un report leggibile, separando input, flotta reale,
    benchmark e scostamenti.
    """
    righe = [
        # Input reali
        ("INPUT AREA REALE", "Area", scenario["area"], ""),
        (
            "INPUT AREA REALE",
            "Clienti reali area",
            scenario["clienti_reali_area"],
            "clienti",
        ),
        (
            "INPUT AREA REALE",
            "Addetti reali area",
            scenario["addetti_reali_area"],
            "addetti",
        ),
        (
            "INPUT AREA REALE",
            "Clienti reali per addetto",
            scenario["clienti_reali_per_addetto"],
            "clienti/addetto",
        ),
        (
            "INPUT SCENARIO",
            "Km annui per addetto",
            scenario["km_annui_addetto"],
            "km/anno",
        ),
        (
            "INPUT SCENARIO",
            "Clienti benchmark per addetto",
            scenario["clienti_benchmark_addetto"],
            "clienti/addetto",
        ),
        (
            "INPUT SCENARIO",
            "Frequenza media mensile",
            scenario["frequenza_media_mese"],
            "rifornimenti/cliente/mese",
        ),
        (
            "INPUT SCENARIO",
            "Valore annuo cliente",
            scenario["valore_annuo_cliente"],
            "€/cliente/anno",
        ),
        (
            "INPUT SCENARIO",
            "Costo totale per km",
            scenario["costo_km_totale"],
            "€/km",
        ),

        # Risultati reali
        (
            "RISULTATI REALI",
            "Valore annuo totale clienti area",
            scenario["valore_annuo_totale_clienti"],
            "€/anno",
        ),
        (
            "RISULTATI REALI",
            "Valore annuo reale per addetto",
            scenario["valore_annuo_reale_per_addetto"],
            "€/addetto/anno",
        ),
        (
            "RISULTATI REALI",
            "Km annui flotta reale",
            scenario["km_annui_flotta_reale"],
            "km/anno",
        ),
        (
            "RISULTATI REALI",
            "Rifornimenti mese area reale",
            scenario["rifornimenti_mese_area_reale"],
            "rifornimenti/mese",
        ),
        (
            "RISULTATI REALI",
            "Rifornimenti mese reali per addetto",
            scenario["rifornimenti_mese_reali_per_addetto"],
            "rifornimenti/addetto/mese",
        ),
        (
            "RISULTATI REALI",
            "Rifornimenti giorno reali per addetto",
            scenario["rifornimenti_giorno_reali_per_addetto"],
            "rifornimenti/addetto/giorno",
        ),
        (
            "RISULTATI REALI",
            "Km reali per rifornimento",
            scenario["km_reali_per_rifornimento"],
            "km/rifornimento",
        ),
        (
            "RISULTATI REALI",
            "Costo logistico annuo flotta reale",
            scenario["costo_logistico_annuo_flotta_reale"],
            "€/anno",
        ),
        (
            "RISULTATI REALI",
            "Costo logistico annuo reale per addetto",
            scenario["costo_logistico_annuo_reale_per_addetto"],
            "€/addetto/anno",
        ),
        (
            "RISULTATI REALI",
            "Costo logistico annuo reale per cliente",
            scenario["costo_logistico_annuo_reale_per_cliente"],
            "€/cliente/anno",
        ),
        (
            "RISULTATI REALI",
            "Margine logistico annuo area reale",
            scenario["margine_logistico_annuo_area_reale"],
            "€/anno",
        ),
        (
            "RISULTATI REALI",
            "Margine logistico annuo reale per addetto",
            scenario["margine_logistico_annuo_reale_per_addetto"],
            "€/addetto/anno",
        ),
        (
            "RISULTATI REALI",
            "Margine logistico annuo reale per cliente",
            scenario["margine_logistico_annuo_reale_per_cliente"],
            "€/cliente/anno",
        ),
        (
            "RISULTATI REALI",
            "Incidenza costo logistico area reale",
            scenario["incidenza_costo_logistico_area_reale"],
            "%",
        ),

        # Benchmark
        (
            "BENCHMARK",
            "Addetti benchmark teorici",
            scenario["addetti_benchmark_teorici"],
            "addetti",
        ),
        (
            "BENCHMARK",
            "Addetti benchmark interi",
            scenario["addetti_benchmark_interi"],
            "addetti",
        ),
        (
            "BENCHMARK",
            "Capacità clienti benchmark con addetti reali",
            scenario[
                "capacita_clienti_benchmark_con_addetti_reali"
            ],
            "clienti",
        ),
        (
            "BENCHMARK",
            "Costo logistico benchmark con addetti interi",
            scenario["costo_logistico_benchmark_interi"],
            "€/anno",
        ),
        (
            "BENCHMARK",
            "Valore annuo benchmark per addetto teorico",
            scenario[
                "valore_annuo_benchmark_per_addetto_teorico"
            ],
            "€/addetto/anno",
        ),

        # Scostamenti
        (
            "SCOSTAMENTI",
            "Differenza addetti benchmark vs reali",
            scenario["differenza_addetti_benchmark_vs_reali"],
            "addetti",
        ),
        (
            "SCOSTAMENTI",
            "Differenza clienti/addetto reale vs benchmark",
            scenario[
                "differenza_clienti_per_addetto_reale_vs_benchmark"
            ],
            "clienti/addetto",
        ),
        (
            "SCOSTAMENTI",
            "Differenza costo reale vs benchmark",
            scenario["differenza_costo_reale_vs_benchmark"],
            "€/anno",
        ),
        (
            "SCOSTAMENTI",
            "Risparmio potenziale benchmark",
            scenario["risparmio_potenziale_benchmark"],
            "€/anno",
        ),
    ]

    return pd.DataFrame(
        righe,
        columns=["Sezione", "Indicatore", "Valore", "Unità"],
    )


# =====================================================================
# 5. MATRICI A DOPPIA ENTRATA
# =====================================================================

def crea_matrice_benchmark(
    df: pd.DataFrame,
    variabile_risultato: str,
    scenario_frequenza: str,
    valore_annuo_cliente: float,
    costo_km_totale: float,
) -> pd.DataFrame:
    """
    Righe: km annui per addetto.
    Colonne: clienti benchmark per addetto.
    Celle: indicatore scelto.

    I clienti e gli addetti reali dell'area restano fissi.
    """
    dati = df[
        (df["scenario_frequenza"] == scenario_frequenza)
        & (df["valore_annuo_cliente"] == valore_annuo_cliente)
        & (df["costo_km_totale"] == costo_km_totale)
    ]

    matrice = dati.pivot(
        index="km_annui_addetto",
        columns="clienti_benchmark_addetto",
        values=variabile_risultato,
    )
    matrice.index.name = "Km annui per addetto"
    matrice.columns.name = "Clienti benchmark per addetto"
    return matrice


def crea_matrice_economica_area_reale(
    df: pd.DataFrame,
    variabile_risultato: str,
    km_annui_addetto: float,
    scenario_frequenza: str,
    clienti_benchmark_addetto: float,
) -> pd.DataFrame:
    """
    Righe: costo totale per km.
    Colonne: valore annuo del cliente.
    Celle: indicatore economico della FLOTTA REALE.

    Il benchmark è selezionato solo per identificare una riga univoca;
    non modifica i risultati reali dell'area.
    """
    dati = df[
        (df["km_annui_addetto"] == km_annui_addetto)
        & (df["scenario_frequenza"] == scenario_frequenza)
        & (
            df["clienti_benchmark_addetto"]
            == clienti_benchmark_addetto
        )
    ]

    matrice = dati.pivot(
        index="costo_km_totale",
        columns="valore_annuo_cliente",
        values=variabile_risultato,
    )
    matrice.index.name = "Costo totale per km (€)"
    matrice.columns.name = "Valore annuo cliente (€)"
    return matrice


def crea_matrici_standard(
    df: pd.DataFrame,
    scenario_frequenza: str = "F1",
    valore_annuo_cliente: float = 1_000,
    costo_km_totale: float = 0.88,
    km_scenario_economico: float = 50_000,
    clienti_benchmark_scenario_economico: float = 150,
) -> Dict[str, pd.DataFrame]:
    """
    Crea matrici di confronto mantenendo sempre visibili i dati reali
    dell'area regionale.
    """
    matrici: Dict[str, pd.DataFrame] = {}

    indicatori_benchmark = {
        "addetti_benchmark_interi": "addetti_benchmark_interi",
        "delta_addetti_vs_reali": (
            "differenza_addetti_benchmark_vs_reali"
        ),
        "capacita_con_addetti_reali": (
            "capacita_clienti_benchmark_con_addetti_reali"
        ),
        "costo_benchmark_interi": (
            "costo_logistico_benchmark_interi"
        ),
        "delta_costo_reale_benchmark": (
            "differenza_costo_reale_vs_benchmark"
        ),
        "risparmio_potenziale": "risparmio_potenziale_benchmark",
    }

    for nome, indicatore in indicatori_benchmark.items():
        matrici[nome] = crea_matrice_benchmark(
            df=df,
            variabile_risultato=indicatore,
            scenario_frequenza=scenario_frequenza,
            valore_annuo_cliente=valore_annuo_cliente,
            costo_km_totale=costo_km_totale,
        )

    indicatori_area_reale = {
        "costo_flotta_reale": "costo_logistico_annuo_flotta_reale",
        "margine_area_reale": "margine_logistico_annuo_area_reale",
        "incidenza_costo_reale": (
            "incidenza_costo_logistico_area_reale"
        ),
        "valore_reale_per_addetto": "valore_annuo_reale_per_addetto",
        "costo_reale_per_cliente": (
            "costo_logistico_annuo_reale_per_cliente"
        ),
    }

    for nome, indicatore in indicatori_area_reale.items():
        matrici[nome] = crea_matrice_economica_area_reale(
            df=df,
            variabile_risultato=indicatore,
            km_annui_addetto=km_scenario_economico,
            scenario_frequenza=scenario_frequenza,
            clienti_benchmark_addetto=(
                clienti_benchmark_scenario_economico
            ),
        )

    return matrici


# =====================================================================
# 6. ESPORTAZIONE
# =====================================================================

def esporta_excel(
    df: pd.DataFrame,
    report_scelto: pd.DataFrame,
    matrici: Dict[str, pd.DataFrame],
    percorso: str | Path = "risultati_ocs_regionale.xlsx",
) -> Path:
    percorso = Path(percorso)

    parametri = pd.DataFrame(
        {
            "Parametro": [
                "Area",
                "Clienti reali area",
                "Addetti reali area",
                "Clienti reali per addetto",
                "Giorni lavorativi mese",
            ],
            "Valore": [
                NOME_AREA,
                CLIENTI_OCS_REALI,
                ADDETTI_OCS_REALI,
                CLIENTI_OCS_REALI / ADDETTI_OCS_REALI,
                GIORNI_LAVORATIVI_MESE,
            ],
        }
    )

    with pd.ExcelWriter(percorso, engine="openpyxl") as writer:
        report_scelto.to_excel(
            writer,
            sheet_name="Scenario_scelto",
            index=False,
        )
        parametri.to_excel(
            writer,
            sheet_name="Area_reale",
            index=False,
        )
        df.to_excel(
            writer,
            sheet_name="Tutti_scenari",
            index=False,
        )

        nomi_usati: set[str] = set()
        for nome, matrice in matrici.items():
            base = nome[:31]
            foglio = base
            contatore = 1
            while foglio in nomi_usati:
                suffisso = f"_{contatore}"
                foglio = base[: 31 - len(suffisso)] + suffisso
                contatore += 1
            nomi_usati.add(foglio)
            matrice.to_excel(writer, sheet_name=foglio)

    return percorso.resolve()


# =====================================================================
# 7. ESECUZIONE E PUBBLICAZIONE DELLO SCENARIO SCELTO
# =====================================================================

def main() -> None:
    df = genera_tutti_scenari()

    # Scenario scelto da valutare e pubblicare.
    KM_SCELTO = 50_000
    CLIENTI_BENCHMARK_SCELTI = 150
    FREQUENZA_SCELTA = "F1"
    VALORE_CLIENTE_SCELTO = 1_000
    COSTO_KM_SCELTO = 0.88

    scenario = seleziona_scenario(
        df=df,
        km_annui_addetto=KM_SCELTO,
        clienti_benchmark_addetto=CLIENTI_BENCHMARK_SCELTI,
        scenario_frequenza=FREQUENZA_SCELTA,
        valore_annuo_cliente=VALORE_CLIENTE_SCELTO,
        costo_km_totale=COSTO_KM_SCELTO,
    )

    report = crea_report_scenario(scenario)

    matrici = crea_matrici_standard(
        df=df,
        scenario_frequenza=FREQUENZA_SCELTA,
        valore_annuo_cliente=VALORE_CLIENTE_SCELTO,
        costo_km_totale=COSTO_KM_SCELTO,
        km_scenario_economico=KM_SCELTO,
        clienti_benchmark_scenario_economico=(
            CLIENTI_BENCHMARK_SCELTI
        ),
    )

    print("=" * 76)
    print(f"AREA REGIONALE REALE: {NOME_AREA}")
    print("=" * 76)
    print(
        f"Clienti reali: {CLIENTI_OCS_REALI:,} | "
        f"Addetti reali: {ADDETTI_OCS_REALI} | "
        f"Clienti reali/addetto: "
        f"{CLIENTI_OCS_REALI / ADDETTI_OCS_REALI:,.2f}"
    )
    print()

    print("SCENARIO SCELTO E PUBBLICATO")
    print(
        f"Km/addetto: {KM_SCELTO:,.0f} | "
        f"Benchmark clienti/addetto: {CLIENTI_BENCHMARK_SCELTI:,.0f} | "
        f"Frequenza: {FREQUENZA_SCELTA} | "
        f"Valore cliente: € {VALORE_CLIENTE_SCELTO:,.2f} | "
        f"Costo/km: € {COSTO_KM_SCELTO:,.2f}"
    )
    print()

    print("RISULTATI REALI DELL'AREA")
    reali = [
        "valore_annuo_totale_clienti",
        "valore_annuo_reale_per_addetto",
        "km_annui_flotta_reale",
        "rifornimenti_mese_reali_per_addetto",
        "rifornimenti_giorno_reali_per_addetto",
        "km_reali_per_rifornimento",
        "costo_logistico_annuo_flotta_reale",
        "costo_logistico_annuo_reale_per_cliente",
        "margine_logistico_annuo_area_reale",
        "incidenza_costo_logistico_area_reale",
    ]
    print(scenario[reali].to_string())
    print()

    print("CONFRONTO CON IL BENCHMARK")
    benchmark = [
        "addetti_benchmark_teorici",
        "addetti_benchmark_interi",
        "differenza_addetti_benchmark_vs_reali",
        "capacita_clienti_benchmark_con_addetti_reali",
        "costo_logistico_benchmark_interi",
        "differenza_costo_reale_vs_benchmark",
        "risparmio_potenziale_benchmark",
    ]
    print(scenario[benchmark].to_string())
    print()

    file_excel = esporta_excel(
        df=df,
        report_scelto=report,
        matrici=matrici,
        percorso="risultati_ocs_regionale.xlsx",
    )

    print(f"Combinazioni generate: {len(df):,}")
    print(f"File Excel creato: {file_excel}")


if __name__ == "__main__":
    main()
'''

path = Path("/mnt/data/simulatore_ocs_regionale.py")
path.write_text(code, encoding="utf-8")
print(path)
print(f"Righe: {len(code.splitlines())}")

/mnt/data/simulatore_ocs_regionale.py
Righe: 1033


In [ ]:
import subprocess, sys, os, pathlib

result = subprocess.run(
    [sys.executable, "/mnt/data/simulatore_ocs_regionale.py"],
    cwd="/mnt/data",
    capture_output=True,
    text=True,
    timeout=120,
)
print(result.stdout)
print(result.stderr)
print("returncode:", result.returncode)

AREA REGIONALE REALE: Marche
Clienti reali: 600 | Addetti reali: 3 | Clienti reali/addetto: 200.00

SCENARIO SCELTO E PUBBLICATO
Km/addetto: 50,000 | Benchmark clienti/addetto: 150 | Frequenza: F1 | Valore cliente: € 1,000.00 | Costo/km: € 0.88

RISULTATI REALI DELL'AREA
valore_annuo_totale_clienti                   600000
valore_annuo_reale_per_addetto              200000.0
km_annui_flotta_reale                         150000
rifornimenti_mese_reali_per_addetto            200.0
rifornimenti_giorno_reali_per_addetto       9.090909
km_reali_per_rifornimento                  20.833333
costo_logistico_annuo_flotta_reale          132000.0
costo_logistico_annuo_reale_per_cliente        220.0
margine_logistico_annuo_area_reale          468000.0
incidenza_costo_logistico_area_reale            0.22

CONFRONTO CON IL BENCHMARK
addetti_benchmark_teorici                            4.0
addetti_benchmark_interi                               4
differenza_addetti_benchmark_vs_reali                  1